# ANC55 / ANC56 with P3HO₄ / P3HO₈

**To run:** select **ipha_clean**, set **`RUN_ANALYSIS=True`** in the first code cell, then click **Run All**. That calculates missing data and saves Figures 1–6, comparison figures and residue-ranking tables. Existing compatible calculations are reused and existing results are protected. No analysis has been run during notebook preparation.

For later redraws, set **`RUN_ANALYSIS=False`** and click **Run All**: plots and tables use the saved CSVs. Representative PDBs are optional: set `EXPORT_REPRESENTATIVE_FRAMES=True` if you want them.

The methods and plotting style come from `enzyme_pha_analysis.ipynb`. System order is ANC55/P3HO₄, ANC55/P3HO₈, ANC56/P3HO₄, ANC56/P3HO₈; P3HO₄ is blue and P3HO₈ orange. All protein residues are analysed; **SER138, ASP194 and HSD226** are highlighted.

Total energy and minimum distance alone cannot establish binding affinity. Energy depends on system composition, and the longer P3HO₈ polymer has more atoms and contact opportunities. Treat these as descriptive comparisons of individual trajectories.


## 1. Settings

The paths, labels, colours, sampling, time window and active-site residues are already configured below. For the first run, you only need to change **`RUN_ANALYSIS=False` to `True`**. GROMACS (`gmx` or `gmx_mpi`) must be on `PATH`.

Defaults: **0–1000 ns**, every stored frame (**STRIDE=1**), **3.5 Å** residue contacts, with 3.0 Å as a sensitivity cutoff. Actual coverage and spacing are checked when calculating. Distances use the original periodic box and heavy atoms with TPR mass >3 u; RMSD aligns protein N/CA/C to the first analysed frame. Units are ns, Å and kJ/mol.

ANC45's established active site is **SER138, ASP194, HSD226**, from `ANC45_P3HO4_binding_modes.ipynb`. Their names/numbers and side-chain atoms were verified in all four production TPRs, and each full atom order matches its `step6.2_npt.gro`. All enzymes have 244 residues and 732 backbone atoms; P3HO₄/P3HO₈ have 41/81 heavy atoms. No additional active-site residues are inferred.

Change `OUTPUT_TAG` only when changing calculation settings. Per-system results save in their own `analysis/`; comparisons save in `PHA_new_Enzyme/analysis/`. Figure filenames get an automatic timestamp so redraws preserve older figures.


In [ ]:
from pathlib import Path
from datetime import datetime
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# FIRST RUN: change this to True, select ipha_clean, then click Run All.
# LATER: False redraws from saved CSVs. Existing calculations are reused.
RUN_ANALYSIS = False
EXPORT_REPRESENTATIVE_FRAMES = False  # Optional: also save representative PDBs.
OUTPUT_TAG = "new_enzyme_20261008_0_1000ns_v1"  # Change only for different analysis settings.

# EDIT: ordered systems and all input filenames.
BASE_DIR = Path("/Users/k20098771/Data/MD_projects/PHA/MD_data/gromacs/PHA_new_Enzyme")
SYSTEMS = ["ANC55_P3HO4_gromacs", "ANC55_P3HO8_gromacs",
           "ANC56_P3HO4_gromacs", "ANC56_P3HO8_gromacs"]
EDR_NAME = "production_combined_1us.edr"  # raw folder
TPR_NAME = "step7_production.tpr"         # raw folder
GRO_NAME = "step6.2_npt.gro"               # raw folder, matching VMD reference
TRAJECTORY_NAME = "processed_protein_centered.xtc"  # analysis/ folder
PROCESSING_RECORD_NAME = "instrcution.txt"  # existing filename, including its spelling

# EDIT: labels and polymer colours (shared by every figure).
SYSTEM_LABELS = {"ANC55_P3HO4_gromacs": "ANC55 · P3HO₄",
                 "ANC55_P3HO8_gromacs": "ANC55 · P3HO₈",
                 "ANC56_P3HO4_gromacs": "ANC56 · P3HO₄",
                 "ANC56_P3HO8_gromacs": "ANC56 · P3HO₈"}
POLYMER_BY_SYSTEM = {s: ("P3HO4" if "P3HO4" in s else "P3HO8") for s in SYSTEMS}
POLYMER_COLOURS = {"P3HO4": "tab:blue", "P3HO8": "tab:orange"}
SYSTEM_COLOURS = {s: POLYMER_COLOURS[POLYMER_BY_SYSTEM[s]] for s in SYSTEMS}
ACTIVE_SITE_COLOUR = "tab:red"
ACTIVE_SITE_RESIDUES = {138: "SER", 194: "ASP", 226: "HSD"}
ACTIVE_SITE_ATOMS = {138: ("OG",), 194: ("OD1", "OD2"), 226: ("ND1", "NE2")}

# EDIT: sampling, inclusive time window, smoothing and distance cutoffs.
STRIDE = 1
START_NS, END_NS = 0.0, 1000.0
ROLLING_WINDOW_FRAMES = 1  # saved values; None omits saved rolling means
PLOT_ROLLING_WINDOW = 100  # sampled frames; only affects plotted rolling means
PROTEIN_SELECTION = "protein"
BACKBONE_SELECTION = "protein and name N CA C"
PHA_SELECTION = "resname LIG"
DISTANCE_CUTOFF_A = 3.5  # Figure 4 global proximity
FIGURE5_CUTOFF_A = 3.5  # Figure 5 contact occupancy
RESIDUE_CUTOFFS_A = (3.5, 3.0)  # Figure 6 contact maps, rethreshold saved distances
CONTACT_CUTOFF_A = 3.5  # rankings / representative-frame selection
PERIOD_LENGTH_NS = 200.0
SUMMARY_WINDOWS_NS = [(float(s), float(min(s + PERIOD_LENGTH_NS, END_NS)))
                      for s in np.arange(START_NS, END_NS, PERIOD_LENGTH_NS)]
SNAPSHOT_TIE_NS = 1e-9
FIGURE3_TOLERANCE_A = 2e-5

# Counts measured from each production TPR, not inferred from the old systems.
EXPECTED_SELECTION_COUNTS = {
    s: {"protein_residues": 244, "backbone_atoms": 732,
        "pha_atoms": 99 if POLYMER_BY_SYSTEM[s] == "P3HO4" else 195,
        "pha_heavy_atoms": 41 if POLYMER_BY_SYSTEM[s] == "P3HO4" else 81}
    for s in SYSTEMS
}
COMPARISON_DIR = BASE_DIR / "analysis"
PLOT_SYSTEMS = list(SYSTEMS)
WITHIN_SYSTEMS = list(SYSTEMS)
RESIDUE_SYSTEMS = list(SYSTEMS)
WITHIN_OUTPUT_DIR = COMPARISON_DIR
RESIDUE_COMPARISON_DIR = COMPARISON_DIR

# Automatic workflow — no additional switches to set.
# Each redraw gets a new figure filename; existing PNGs stay intact.
FIGURE_TAG = datetime.now().strftime("style_%Y%m%d_%H%M%S_%f")
_metric_names = ("total_energy", "protein_backbone_rmsd", "enzyme_pha_min_distance")
_saved_results_available = all(
    (BASE_DIR / system / "analysis" / f"{system.removesuffix('_gromacs')}_{OUTPUT_TAG}_{suffix}").is_file()
    for system in SYSTEMS
    for suffix in [*(f"{metric}.csv" for metric in _metric_names), "summary.json",
                   "residue_pha_distances_A.csv", "residue_pha_residues.csv", "residue_pha_metadata.json"]
)
RUN_RESIDUE_CALCULATION = RUN_ANALYSIS
RUN_PLOTS = RUN_ANALYSIS or _saved_results_available
RUN_PERIOD_TABLES = RUN_PLOTS
SAVE_FIGURES = RUN_PLOTS
SAVE_WITHIN_COMPARISON = RUN_PLOTS
SAVE_RESIDUE_FIGURES = RUN_PLOTS
SAVE_PERIOD_RESULTS = RUN_PERIOD_TABLES
OVERWRITE = False
OVERWRITE_RESIDUE_DATA = False
OVERWRITE_DERIVED_OUTPUTS = False
if not RUN_PLOTS:
    print("First run: set RUN_ANALYSIS=True in this cell, then click Run All.")
elif RUN_ANALYSIS:
    print("Calculating missing data, then saving Figures 1–6 and residue tables. Existing compatible data are reused.")
else:
    print("Redrawing Figures 1–6 and residue tables from saved CSVs.")

if type(STRIDE) is not int or STRIDE < 1:
    raise ValueError("STRIDE must be a positive integer.")
if not np.isfinite([START_NS, END_NS]).all() or not 0 <= START_NS < END_NS:
    raise ValueError("Require 0 <= START_NS < END_NS, in ns.")
for value in (ROLLING_WINDOW_FRAMES, PLOT_ROLLING_WINDOW):
    if value is not None and (type(value) is not int or value < 1):
        raise ValueError("Rolling windows must be positive integers or None.")
for tag in (OUTPUT_TAG, FIGURE_TAG):
    if not tag or any(c not in "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_-" for c in tag):
        raise ValueError("Tags must contain only letters, numbers, underscores or hyphens.")
if not ACTIVE_SITE_RESIDUES or set(ACTIVE_SITE_RESIDUES) != set(ACTIVE_SITE_ATOMS):
    raise ValueError("Provide the established active-site definition and matching atom names.")
if not np.isfinite(PERIOD_LENGTH_NS) or PERIOD_LENGTH_NS <= 0:
    raise ValueError("PERIOD_LENGTH_NS must be positive.")
if any(not np.isfinite(c) or c <= 0 for c in (DISTANCE_CUTOFF_A, FIGURE5_CUTOFF_A, CONTACT_CUTOFF_A, *RESIDUE_CUTOFFS_A)):
    raise ValueError("Cutoffs must be positive finite Å values.")


In [ ]:
# Helpers do not open files until explicitly called by enabled sections.
def system_inputs(system):
    folder = BASE_DIR / system
    return {"edr": folder / EDR_NAME, "tpr": folder / TPR_NAME,
            "xtc": folder / "analysis" / TRAJECTORY_NAME, "gro": folder / GRO_NAME}


def validate_active_site(protein, system):
    result = []
    for resid, resname in ACTIVE_SITE_RESIDUES.items():
        hit = protein.select_atoms(f"resid {resid}").residues
        if len(hit) != 1 or hit[0].resname != resname:
            raise ValueError(f"{system}: expected unique {resname}{resid}; found {list(hit.resnames)}. No numbering is inferred.")
        for name in ACTIVE_SITE_ATOMS[resid]:
            if np.count_nonzero(hit.atoms.names == name) != 1:
                raise ValueError(f"{system}: {resname}{resid} missing unique atom {name}.")
        result.append({"resid": int(resid), "resname": resname, "segid": hit[0].segid,
                       "required_atoms": list(ACTIVE_SITE_ATOMS[resid])})
    return result


def validate_topology_reference(system):
    # TPR/GRO topology parsers only: no trajectory or coordinate calculations.
    import MDAnalysis as mda
    from MDAnalysis.topology.TPRParser import TPRParser
    from MDAnalysis.topology.GROParser import GROParser
    paths = system_inputs(system)
    tpr = mda.Universe(TPRParser(str(paths["tpr"])).parse())
    gro = mda.Universe(GROParser(str(paths["gro"])).parse())
    for attribute in ("names", "resnames", "resids"):
        if not np.array_equal(getattr(tpr.atoms, attribute), getattr(gro.atoms, attribute)):
            raise ValueError(f"{system}: GRO/TPR atom ordering differs ({attribute}).")
    protein = tpr.select_atoms(PROTEIN_SELECTION)
    pha = tpr.select_atoms(PHA_SELECTION)
    counts = {"protein_residues": len(protein.residues),
              "backbone_atoms": len(tpr.select_atoms(BACKBONE_SELECTION)),
              "pha_atoms": len(pha), "pha_heavy_atoms": int((pha.masses > 3).sum())}
    if counts != EXPECTED_SELECTION_COUNTS[system] or len(pha.residues) != 1:
        raise ValueError(f"{system}: unexpected topology counts: {counts}.")
    return {"gro_tpr_atom_order_matches": True, "system_atoms": len(tpr.atoms),
            "active_site": validate_active_site(protein, system), "counts": counts}


def mark_active_site(residues):
    for resid, resname in ACTIVE_SITE_RESIDUES.items():
        hit = residues.loc[residues.resid == resid]
        if len(hit) != 1 or hit.resname.iloc[0] != resname:
            raise ValueError(f"Saved residue table does not contain unique {resname}{resid}.")
    table = residues.copy()
    table["active_site"] = table.resid.isin(ACTIVE_SITE_RESIDUES)
    return table


def highlight_table(table):
    return table.style.apply(lambda row: ["color: red; font-weight: bold" if row.active_site else ""
                                         for _ in row], axis=1)


def spacing_report(times):
    times = np.asarray(times, dtype=float)
    if len(times) < 2 or not np.isfinite(times).all() or np.any(np.diff(times) <= 0):
        raise ValueError("Need at least two finite, strictly increasing timestamps.")
    steps = np.diff(times)
    return {"frames": len(times), "start_ns": float(times[0]), "end_ns": float(times[-1]),
            "min_spacing_ns": float(steps.min()), "max_spacing_ns": float(steps.max()),
            "median_spacing_ns": float(np.median(steps)),
            "uniform": bool(np.allclose(steps, steps[0], atol=1e-8, rtol=0))}


def write_csv_protected(table, path, **kwargs):
    path = Path(path)
    content = table.to_csv(index=False, **kwargs)
    if path.is_symlink() or path.parent.is_symlink():
        raise ValueError(f"Refuse a symlink output: {path}")
    if path.exists():
        if path.read_text() == content:
            return  # Reusing identical derived tables is safe.
        if not OVERWRITE_DERIVED_OUTPUTS:
            raise FileExistsError(f"Protected result: {path}; choose a new OUTPUT_TAG.")
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w" if OVERWRITE_DERIVED_OUTPUTS else "x") as handle:
        handle.write(content)


def save_figure_protected(fig, path, **kwargs):
    path = Path(path)
    path = path.with_name(f"{path.stem}_{FIGURE_TAG}{path.suffix}")
    if path.is_symlink() or path.parent.is_symlink():
        raise ValueError(f"Refuse a symlink output: {path}")
    if path.exists() and not OVERWRITE_DERIVED_OUTPUTS:
        raise FileExistsError(f"Protected figure: {path}; use a fresh FIGURE_TAG or preview with saving disabled.")
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("wb" if OVERWRITE_DERIVED_OUTPUTS else "xb") as handle:
        fig.savefig(handle, format=path.suffix.lstrip("."), **kwargs)
    print("Saved figure:", path)
    return path


def validate_saved_metrics(system, paths, prefix):
    summary = json.loads(Path(f"{prefix}_summary.json").read_text())
    if (summary["analysis_window_ns"] != [START_NS, END_NS] or summary["stride"] != STRIDE
            or summary["selections"] != {"protein": PROTEIN_SELECTION, "backbone": BACKBONE_SELECTION,
                                          "pha": PHA_SELECTION, "heavy": "TPR mass > 3 u"}):
        raise ValueError(f"{system}: saved analysis settings differ; choose a new OUTPUT_TAG.")
    expected_site = [(int(resid), resname) for resid, resname in ACTIVE_SITE_RESIDUES.items()]
    if [(r["resid"], r["resname"]) for r in summary["active_site"]] != expected_site:
        raise ValueError(f"{system}: saved active-site definition differs; choose a new OUTPUT_TAG.")
    for key, path in paths.items():
        if (Path(summary["inputs"][key]).resolve() != path.resolve()
                or summary["input_size_mtime_ns"][key] != [path.stat().st_size, path.stat().st_mtime_ns]):
            raise ValueError(f"{system}: input {key} changed; choose a new OUTPUT_TAG.")
    for metric, column in [("total_energy", "total_energy_kj_mol"),
                           ("protein_backbone_rmsd", "protein_backbone_rmsd_A"),
                           ("enzyme_pha_min_distance", "enzyme_pha_min_distance_A")]:
        data = pd.read_csv(f"{prefix}_{metric}.csv")
        spacing_report(data.time_ns)
        if (not np.isfinite(data[column]).all()
                or not data.time_ns.between(START_NS, END_NS).all()
                or not np.allclose([data.time_ns.iloc[0], data.time_ns.iloc[-1]],
                                   [START_NS, END_NS], atol=1e-8, rtol=0)):
            raise ValueError(f"{system}: invalid saved {metric}; existing files remain protected.")
        if metric != "total_energy":
            if (not np.equal(data.frame, np.floor(data.frame)).all()
                    or data.frame.iloc[0] != summary["reference_frame"]
                    or not (np.diff(data.frame) == STRIDE).all()):
                raise ValueError(f"{system}: saved frame sampling differs; choose a new OUTPUT_TAG.")


## 2. Calculation checks (only when RUN_ANALYSIS=True)

RMSD is calculated on coordinate copies with equal atom weights and optimal translation/rotation. This is equivalent to backbone alignment followed by backbone RMSD, without modifying the trajectory. The known-coordinate checks below test rotation/translation removal, Å units, and minimum-image distance across a periodic boundary.

In [ ]:
if RUN_ANALYSIS:
    import shutil
    import subprocess
    from tempfile import TemporaryDirectory

    import MDAnalysis as mda
    from MDAnalysis.analysis.rms import rmsd
    from MDAnalysis.lib.distances import distance_array
    from MDAnalysis.lib.mdamath import triclinic_vectors
    reference_test = np.array([[0., 0., 0.], [1., 0., 0.], [0., 2., 0.], [0., 0., 3.]])
    rotation_test = np.array([[0., -1., 0.], [1., 0., 0.], [0., 0., 1.]])
    mobile_test = reference_test @ rotation_test + [4., -3., 2.]
    assert rmsd(mobile_test.copy(), reference_test.copy(), superposition=True) < 1e-6
    pair_test = distance_array(np.array([[0.2, 0., 0.]]), np.array([[9.8, 0., 0.], [5., 0., 0.]]),
                               box=np.array([10., 10., 10., 90., 90., 90.]))
    assert np.isclose(pair_test.min(), 0.4, atol=1e-6)
    assert not np.isclose(pair_test.mean(), pair_test.min())
    print("Alignment in Å and periodic minimum distance in Å passed.")
else:
    print("RUN_ANALYSIS=False; trajectory calculation checks disabled.")


## 3. Data export helpers

`gmx energy` selects **Total-Energy by name**. Energy CSVs keep every extracted sample within the configured inclusive time window. These helpers calculate/export data only; plotting is below.

Heavy atoms are selected using TPR masses >3 u, excluding hydrogens/deuterium/tritium and zero-mass sites. The residue calculation validates that retained heavy-atom names begin with C/N/O/S and masses are finite. Selections, counts, input paths, sampling and validation details are saved in each summary.

In [ ]:
def extract_energy(edr, log_path):
    with TemporaryDirectory(prefix="ipha_energy_") as scratch:
        xvg = Path(scratch) / "energy.xvg"
        command = [GMX, "energy", "-f", str(edr), "-o", str(xvg), "-dp", "-xvg", "none",
                   "-b", str(START_NS * 1000), "-e", str(END_NS * 1000)]
        result = subprocess.run(command, input="Total-Energy\n0\n", text=True,
                                capture_output=True, cwd=scratch)
        log_path.write_text("Command: " + repr(command) + "\nSelection: Total-Energy\n"
                            + result.stdout + "\n" + result.stderr)
        if result.returncode != 0:
            raise RuntimeError(f"Energy extraction failed; see {log_path}")
        data = np.loadtxt(xvg, ndmin=2)
    if data.shape[1] != 2 or not len(data) or not np.isfinite(data).all():
        raise ValueError("Expected exactly one finite energy term plus time.")
    if np.any(np.diff(data[:, 0]) <= 0):
        raise ValueError("EDR times are not strictly increasing; inspect the supplied merged EDR.")
    energy = pd.DataFrame({"time_ns": data[:, 0] / 1000,
                           "total_energy_kj_mol": data[:, 1]})
    energy = energy.loc[energy.time_ns.between(START_NS, END_NS)].copy()
    if energy.empty or energy.time_ns.iloc[0] > START_NS + 1e-8 or energy.time_ns.iloc[-1] < END_NS - 1e-8:
        raise ValueError("EDR does not cover the full requested window.")
    energy["cumulative_mean_kj_mol"] = energy.total_energy_kj_mol.expanding().mean()
    return energy


def save_data(table, column, prefix, metric):
    data = table.copy()
    if data.empty or not data.time_ns.between(START_NS, END_NS).all():
        raise ValueError("Export data must lie inside the analysis window.")
    if metric != "total_energy" and ROLLING_WINDOW_FRAMES is not None:
        data[f"{column}_rolling_mean"] = data[column].rolling(ROLLING_WINDOW_FRAMES, min_periods=1).mean()
    data.to_csv(f"{prefix}_{metric}.csv", index=False)


## 4. Calculate and save CSVs (only when RUN_ANALYSIS=True)

For each system, check topology/trajectory counts and the printed selections. Protein bonds are checked in the unaligned sampled coordinates to reject a broken protein before RMSD; the notebook does not unwrap, centre or fit the stored trajectory.

The distance is `min(distance_array(protein_heavy, pha_heavy, box=ts.dimensions))` on **unaligned** coordinates. RMSD then uses independent backbone copies. For three sampled frames per system, a separate lattice-image distance check and NumPy SVD alignment cross-check validate the calculations. Only sampled frames are decoded for analysis; unanalysed frames are not claimed to have been validated. XTC has no atom identities, so matching atom counts alone cannot establish topology provenance.

The supplied trajectories were prepared without rotational fitting. Each frame's original box is used; minimum distances cannot diagnose binding affinity or establish convergence.

In [ ]:
if RUN_ANALYSIS:
    GMX = shutil.which("gmx") or shutil.which("gmx_mpi")
    if GMX is None:
        raise RuntimeError("Load GROMACS so gmx or gmx_mpi is on PATH.")
    METRICS = ("total_energy", "protein_backbone_rmsd", "enzyme_pha_min_distance")
    inputs = {}
    pending = []
    for system in SYSTEMS:
        folder = BASE_DIR / system
        inputs[system] = system_inputs(system)
        for path in inputs[system].values():
            if not path.is_file():
                raise FileNotFoundError(path)
        prefix = folder / "analysis" / f"{system.removesuffix('_gromacs')}_{OUTPUT_TAG}"
        targets = [Path(f"{prefix}_{metric}.{ext}") for metric in METRICS for ext in ("csv",)]
        targets += [Path(f"{prefix}_summary.json"), Path(f"{prefix}_energy.log")]
        present = [p.exists() or p.is_symlink() for p in targets]
        if not OVERWRITE and any(present):
            if not all(present):
                raise FileExistsError(f"{system}: incomplete previous results; choose a new OUTPUT_TAG. Existing files remain protected.")
            validate_saved_metrics(system, inputs[system], prefix)
            print(f"{system}: compatible saved metric CSVs found; reusing them.")
        else:
            pending.append(system)
        print(system, "→", inputs[system]["xtc"])

    topology_checks = {system: validate_topology_reference(system) for system in SYSTEMS}
    run_summary = []
    for system in pending:
        print(f"\nAnalysing {system}", flush=True)
        paths = inputs[system]
        before = {key: (p.stat().st_size, p.stat().st_mtime_ns) for key, p in paths.items()}
        label = system.removesuffix("_gromacs")
        prefix = BASE_DIR / system / "analysis" / f"{label}_{OUTPUT_TAG}"
        # Scratch symlink keeps MDAnalysis XTC offset caches out of the simulation folder.
        with TemporaryDirectory(prefix="ipha_research_") as scratch:
            linked_xtc = Path(scratch) / "trajectory.xtc"
            linked_xtc.symlink_to(paths["xtc"])
            u = mda.Universe(str(paths["tpr"]), str(linked_xtc))
            try:
                if u.atoms.n_atoms != u.trajectory.n_atoms:
                    raise ValueError(f"Atom count mismatch in {system}")
                protein = u.select_atoms(PROTEIN_SELECTION)
                backbone = u.select_atoms(BACKBONE_SELECTION)
                pha = u.select_atoms(PHA_SELECTION)
                protein_heavy = protein[protein.masses > 3]
                pha_heavy = pha[pha.masses > 3]
                active_site = validate_active_site(protein, system)
                expected = EXPECTED_SELECTION_COUNTS[system]
                if (len(pha.residues) != 1 or len(pha_heavy) != expected["pha_heavy_atoms"]
                        or len(pha) != expected["pha_atoms"] or len(backbone) != expected["backbone_atoms"]
                        or len(protein.residues) != expected["protein_residues"]):
                    raise ValueError(f"Unexpected selection counts in {system}; inspect topology.")
                if not len(protein_heavy) or np.intersect1d(protein.indices, pha.indices).size:
                    raise ValueError("Empty protein selection or protein/PHA overlap.")
                if len(protein.fragments) != 1:
                    raise ValueError("This notebook expects one connected enzyme molecule.")
                # Inspect all actual stored timestamps; no nominal dt or 1 µs assumption.
                stored_times_ns = np.array([float(ts.time) / 1000 for ts in u.trajectory])
                stored_spacing = spacing_report(stored_times_ns)
                first_time_ns, last_time_ns = stored_times_ns[[0, -1]]
                if first_time_ns > START_NS + 1e-8 or last_time_ns < END_NS - 1e-8:
                    raise ValueError(f"{system} does not cover {START_NS}–{END_NS} ns: {stored_spacing}")
                selected_frames = np.flatnonzero((stored_times_ns >= START_NS - 1e-8)
                                                & (stored_times_ns <= END_NS + 1e-8))[::STRIDE].tolist()
                sampled_spacing = spacing_report(stored_times_ns[selected_frames])
                if not np.allclose(stored_times_ns[[selected_frames[0], selected_frames[-1]]],
                                   [START_NS, END_NS], atol=1e-8, rtol=0):
                    raise ValueError("Requested endpoints must be sampled timestamps; adjust window/STRIDE. No extrapolation.")
                print("Actual stored coverage/spacing:", stored_spacing, flush=True)
                print("Actual sampled coverage/spacing:", sampled_spacing, flush=True)
                u.trajectory[selected_frames[0]]
                reference = backbone.positions.astype(np.float64, copy=True)
                protein_bonds = protein.bonds.indices
                counts = {"system_atoms": len(u.atoms), "protein_residues": len(protein.residues),
                          "protein_atoms": len(protein), "backbone_atoms": len(backbone),
                          "protein_heavy_atoms": len(protein_heavy), "pha_atoms": len(pha),
                          "pha_heavy_atoms": len(pha_heavy),
                          "pha_identity": [{"resid": int(r.resid), "resname": r.resname, "segid": r.segid}
                                           for r in pha.residues]}
                print(counts, flush=True)
                print(f"{len(u.trajectory)} stored frames; {len(selected_frames)} sampled; "
                      f"{first_time_ns:g}–{last_time_ns:g} ns available", flush=True)
                crosscheck_frames = {selected_frames[0], selected_frames[len(selected_frames)//2], selected_frames[-1]}
                rows, crosschecks = [], []
                max_bond_nm = 0.0
                for frame in selected_frames:
                    ts = u.trajectory[frame]
                    box = ts.dimensions.copy() if ts.dimensions is not None else None
                    if box is None or not np.isfinite(box).all() or np.any(box[:3] <= 0) or np.linalg.det(triclinic_vectors(box)) <= 0:
                        raise ValueError(f"Invalid periodic box at frame {frame}")
                    enzyme_xyz = protein_heavy.positions.copy()
                    pha_xyz = pha_heavy.positions.copy()
                    mobile = backbone.positions.astype(np.float64, copy=True)
                    if not all(np.isfinite(x).all() for x in (enzyme_xyz, pha_xyz, mobile)) or not np.isfinite(ts.time):
                        raise ValueError(f"Non-finite sampled coordinates/time at frame {frame}")
                    bond_nm = np.linalg.norm(ts.positions[protein_bonds[:, 0]] - ts.positions[protein_bonds[:, 1]], axis=1) / 10
                    max_bond_nm = max(max_bond_nm, float(bond_nm.max()))
                    if np.any(bond_nm > 0.3) or np.any(bond_nm < 0.05):
                        raise ValueError(f"Protein bond discontinuity at frame {frame}; inspect preparation before RMSD.")
                    # PBC distance first; never align/transform u or its box.
                    minimum_A = float(distance_array(enzyme_xyz, pha_xyz, box=box).min())
                    backbone_A = float(rmsd(mobile.copy(), reference.copy(), superposition=True))
                    if not np.array_equal(backbone.positions, mobile.astype(np.float32)) or not np.array_equal(ts.dimensions, box):
                        raise AssertionError("Coordinates/box changed during the calculation.")
                    rows.append((frame, ts.time / 1000, backbone_A, minimum_A))
                    if frame in crosscheck_frames:
                        # Retain the source notebook's independent orthorhombic image check.
                        if not np.allclose(box[3:], 90, atol=1e-4):
                            raise ValueError("Independent check requires orthorhombic boxes; inspect any non-orthorhombic input before adapting this check.")
                        delta = enzyme_xyz.astype(float)[:, None, :] - pha_xyz.astype(float)[None, :, :]
                        delta -= box[:3] * np.rint(delta / box[:3])
                        independent_min_A = np.linalg.norm(delta, axis=2).min()
                        a, b = mobile - mobile.mean(axis=0), reference - reference.mean(axis=0)
                        left, _, right = np.linalg.svd(a.T @ b)
                        correction = np.diag([1., 1., np.linalg.det(left @ right)])
                        aligned = a @ left @ correction @ right
                        independent_rmsd_A = np.sqrt(np.mean(np.sum((aligned - b)**2, axis=1)))
                        assert np.isclose(minimum_A, independent_min_A, atol=2e-5, rtol=0)
                        assert np.isclose(backbone_A, independent_rmsd_A, atol=2e-5, rtol=0)
                        crosschecks.append({"frame": frame, "distance_error_A": abs(minimum_A-independent_min_A),
                                            "rmsd_error_A": abs(backbone_A-independent_rmsd_A)})
                metrics = pd.DataFrame(rows, columns=["frame", "time_ns", "protein_backbone_rmsd_A", "enzyme_pha_min_distance_A"])
                if not np.isfinite(metrics.to_numpy()).all() or np.any(np.diff(metrics.time_ns) <= 0):
                    raise ValueError("Invalid or non-increasing sampled trajectory data.")
                assert metrics.protein_backbone_rmsd_A.iloc[0] < 1e-5
                trajectory_frames = len(u.trajectory)
            finally:
                u.trajectory.close()
        energy = extract_energy(paths["edr"], Path(f"{prefix}_energy.log"))
        # All energy and trajectory analysis is restricted to the common window.
        energy_range = [float(energy.time_ns.iloc[0]), float(energy.time_ns.iloc[-1])]
        save_data(energy, "total_energy_kj_mol", prefix, "total_energy")
        save_data(metrics[["frame", "time_ns", "protein_backbone_rmsd_A"]],
                  "protein_backbone_rmsd_A", prefix, "protein_backbone_rmsd")
        save_data(metrics[["frame", "time_ns", "enzyme_pha_min_distance_A"]],
                  "enzyme_pha_min_distance_A", prefix, "enzyme_pha_min_distance")
        after = {key: (p.stat().st_size, p.stat().st_mtime_ns) for key, p in paths.items()}
        if before != after:
            raise RuntimeError("An input changed during analysis.")
        summary = {"system": system, "inputs": {k: str(v) for k,v in paths.items()},
                   "input_size_mtime_ns": before, "selections": {"protein": PROTEIN_SELECTION,
                   "backbone": BACKBONE_SELECTION, "pha": PHA_SELECTION, "heavy": "TPR mass > 3 u"},
                   "active_site": active_site, "topology_reference": topology_checks[system],
                   "stored_timestamps": stored_spacing, "sampled_timestamps": sampled_spacing,
                   "energy_timestamps": spacing_report(energy.time_ns),
                   "counts": counts, "trajectory_frames": trajectory_frames, "analysed_frames": len(metrics),
                   "stride": STRIDE, "reference_frame": selected_frames[0],
                   "analysis_window_ns": [START_NS, END_NS], "rolling_window_samples": ROLLING_WINDOW_FRAMES, "trajectory_range_ns": [first_time_ns, last_time_ns],
                   "sampled_range_ns": [float(metrics.time_ns.iloc[0]), float(metrics.time_ns.iloc[-1])],
                   "energy_samples": len(energy), "energy_range_ns": energy_range,
                   "max_sampled_protein_bond_nm": max_bond_nm, "independent_checks": crosschecks,
                   "MDAnalysis_version": mda.__version__, "units": {"time": "ns", "backbone_rmsd": "Å", "minimum_distance": "Å", "energy": "kJ/mol"}}
        Path(f"{prefix}_summary.json").write_text(json.dumps(summary, indent=2) + "\n")
        run_summary.append({"system": label, "frames": len(metrics), "analysed_start_ns": float(metrics.time_ns.iloc[0]),
                            "analysed_end_ns": float(metrics.time_ns.iloc[-1]),
                            "energy_end_ns": energy_range[1], "output_prefix": str(prefix)})
        print(f"Saved three CSVs, log and summary: {prefix}_*", flush=True)
        del energy, metrics

    display(pd.DataFrame(run_summary))
else:
    print("Analysis skipped. The plotting cells below read saved CSVs only.")


## 5. Plot style

Run All draws and saves Figures 1–6 automatically after calculation, or from saved CSVs when `RUN_ANALYSIS=False`. Edit colours in the first settings cell and size/smoothing below. Each redraw saves figures with a new timestamp, preserving earlier images.


In [ ]:
# EDIT: same plotting style as the source notebook.
FIGSIZE = (8, 4.5)
PNG_DPI = 400
PANEL_FIGSIZE = (2 * FIGSIZE[0], 2 * FIGSIZE[1])

if PLOT_ROLLING_WINDOW is not None and (type(PLOT_ROLLING_WINDOW) is not int or PLOT_ROLLING_WINDOW < 1):
    raise ValueError("PLOT_ROLLING_WINDOW must be positive or None.")
plt.rcdefaults()
plt.rcParams.update({"figure.figsize": FIGSIZE, "savefig.dpi": PNG_DPI,
                     "axes.spines.left": True, "axes.spines.bottom": True,
                     "axes.spines.top": False, "axes.spines.right": False})


def load_plot_data(system, metric, column):
    prefix = BASE_DIR / system / "analysis" / f"{system.removesuffix('_gromacs')}_{OUTPUT_TAG}"
    path = Path(f"{prefix}_{metric}.csv")
    if not path.is_file():
        raise FileNotFoundError(f"{path}: calculate this OUTPUT_TAG first or select an existing one.")
    data = pd.read_csv(path)
    values = data[["time_ns", column]].to_numpy()  # Unit-specific column names reject older nm CSVs.
    if data.empty or not np.isfinite(values).all() or np.any(np.diff(data.time_ns) <= 0):
        raise ValueError(f"Invalid plot data: {path}")
    if not data.time_ns.between(START_NS, END_NS).all():
        raise ValueError(f"CSV time window differs from settings: {path}")
    summary = json.loads(Path(f"{prefix}_summary.json").read_text())
    if (summary["analysis_window_ns"] != [START_NS, END_NS] or summary["stride"] != STRIDE
            or summary["selections"]["protein"] != PROTEIN_SELECTION
            or summary["selections"]["backbone"] != BACKBONE_SELECTION
            or summary["selections"]["pha"] != PHA_SELECTION):
        raise ValueError(f"Saved calculation settings differ: {path}")
    if not np.allclose([data.time_ns.iloc[0], data.time_ns.iloc[-1]],
                       [START_NS, END_NS], atol=1e-8, rtol=0):
        raise ValueError(f"Saved CSV must cover both window endpoints: {path}")
    return data, prefix


### Figure 1 — total energy

In [ ]:
if RUN_PLOTS:
    # Figure 1 — edit colours, labels, line widths and axes here, then rerun this cell.
    for system in PLOT_SYSTEMS:
        data, prefix = load_plot_data(system, "total_energy", "total_energy_kj_mol")
        fig, ax = plt.subplots(figsize=FIGSIZE)
        ax.plot(data.time_ns, data.total_energy_kj_mol, color=SYSTEM_COLOURS[system], linewidth=1.0,
                label="Total Energy")
        ax.plot(data.time_ns, data.total_energy_kj_mol.expanding().mean(),
                color="tab:red", linestyle="--", linewidth=1.5, label="Cumulative frame mean")
        ax.set(xlabel="Time (ns)", ylabel="Total energy (kJ/mol)",
               title=f"{SYSTEM_LABELS[system]}: Total energy", xlim=(START_NS, END_NS))
        ax.grid(alpha=0.3)
        ax.legend()
        fig.tight_layout()
        if SAVE_FIGURES:
            save_figure_protected(fig, f"{prefix}_total_energy.png", dpi=PNG_DPI)
        display(fig)
        plt.close(fig)
else:
    print("RUN_PLOTS=False; this section is disabled.")


### Figure 2 — protein backbone RMSD

In [ ]:
if RUN_PLOTS:
    # Figure 2 — edit this cell and rerun it independently of the analysis.
    for system in PLOT_SYSTEMS:
        data, prefix = load_plot_data(system, "protein_backbone_rmsd", "protein_backbone_rmsd_A")
        values = data["protein_backbone_rmsd_A"]
        fig, ax = plt.subplots(figsize=FIGSIZE)
        ax.plot(data.time_ns, values, color=SYSTEM_COLOURS[system], linewidth=0.8, alpha=0.55,
                label="Raw RMSD")
        if PLOT_ROLLING_WINDOW is not None:
            mean = values.rolling(PLOT_ROLLING_WINDOW, min_periods=1).mean()
            ax.plot(data.time_ns, mean, color=SYSTEM_COLOURS[system], linewidth=1.8,
                    label=f"Rolling mean ({PLOT_ROLLING_WINDOW} sampled frames)")
        ax.set(xlabel="Time (ns)", ylabel="Protein backbone RMSD (Å)",
               title=f"{SYSTEM_LABELS[system]}: Protein backbone RMSD", xlim=(START_NS, END_NS))
        ax.grid(alpha=0.3)
        ax.legend()
        fig.tight_layout()
        if SAVE_FIGURES:
            save_figure_protected(fig, f"{prefix}_protein_backbone_rmsd.png", dpi=PNG_DPI)
        final_count = max(1, int(np.ceil(0.20 * len(values))))
        print(f"{system}: mean={values.mean():.4f} Å; SD={values.std():.4f} Å; "
              f"final 20% mean={values.iloc[-final_count:].mean():.4f} Å")
        display(fig)
        plt.close(fig)
else:
    print("RUN_PLOTS=False; this section is disabled.")


### Figure 3 — minimum enzyme–PHA distance

In [ ]:
if RUN_PLOTS:
    # Figure 3 — edit this cell and rerun it independently of the analysis.
    for system in PLOT_SYSTEMS:
        data, prefix = load_plot_data(system, "enzyme_pha_min_distance", "enzyme_pha_min_distance_A")
        values = data["enzyme_pha_min_distance_A"]
        fig, ax = plt.subplots(figsize=FIGSIZE)
        ax.plot(data.time_ns, values, color=SYSTEM_COLOURS[system], linewidth=0.8, alpha=0.55,
                label="Minimum distance (PBC)")
        if PLOT_ROLLING_WINDOW is not None:
            mean = values.rolling(PLOT_ROLLING_WINDOW, min_periods=1).mean()
            ax.plot(data.time_ns, mean, color=SYSTEM_COLOURS[system], linewidth=1.8,
                    label=f"Rolling mean ({PLOT_ROLLING_WINDOW} sampled frames)")
        ax.set(xlabel="Time (ns)", ylabel="Minimum enzyme–PHA distance (Å)",
               title=f"{SYSTEM_LABELS[system]}: Heavy-atom distance with PBC", xlim=(START_NS, END_NS))
        ax.grid(alpha=0.3)
        ax.legend()
        fig.tight_layout()
        if SAVE_FIGURES:
            save_figure_protected(fig, f"{prefix}_enzyme_pha_min_distance.png", dpi=PNG_DPI)
        final_count = max(1, int(np.ceil(0.20 * len(values))))
        print(f"{system}: mean={values.mean():.4f} Å; SD={values.std():.4f} Å; "
              f"final 20% mean={values.iloc[-final_count:].mean():.4f} Å")
        display(fig)
        plt.close(fig)
else:
    print("RUN_PLOTS=False; this section is disabled.")


### Figures 1–3 — four-system comparison panels

The optional comparison panels reuse the same metric CSVs, colours, line styles, axes and system order. They save in the shared comparison folder. Each panel keeps its own y-scale; total energy remains a whole-system diagnostic.

In [ ]:
if RUN_PLOTS:
    metrics_to_plot = [("total_energy", "total_energy_kj_mol", "Total energy (kJ/mol)", 1),
                       ("protein_backbone_rmsd", "protein_backbone_rmsd_A", "Protein backbone RMSD (Å)", 2),
                       ("enzyme_pha_min_distance", "enzyme_pha_min_distance_A", "Minimum enzyme–PHA distance (Å)", 3)]
    for metric, column, ylabel, number in metrics_to_plot:
        fig, axes = plt.subplots(2, 2, figsize=PANEL_FIGSIZE, sharex=True)
        for ax, system in zip(axes.flat, SYSTEMS):
            data, _ = load_plot_data(system, metric, column)
            ax.plot(data.time_ns, data[column], color=SYSTEM_COLOURS[system],
                    linewidth=1.0 if number == 1 else 0.8, alpha=1 if number == 1 else 0.55,
                    label="Total energy" if number == 1 else "Raw sampled values")
            if number == 1:
                ax.plot(data.time_ns, data[column].expanding().mean(), color="tab:red",
                        linestyle="--", linewidth=1.5, label="Cumulative frame mean")
            elif PLOT_ROLLING_WINDOW is not None:
                ax.plot(data.time_ns, data[column].rolling(PLOT_ROLLING_WINDOW, min_periods=1).mean(),
                        color=SYSTEM_COLOURS[system], linewidth=1.8,
                        label=f"Rolling mean ({PLOT_ROLLING_WINDOW} sampled frames)")
            ax.set(title=SYSTEM_LABELS[system], xlabel="Time (ns)", ylabel=ylabel, xlim=(START_NS, END_NS))
            ax.grid(alpha=0.3)
            ax.legend(fontsize=8)
        fig.suptitle(f"Figure {number} — ANC55 / ANC56 with P3HO₄ / P3HO₈")
        fig.tight_layout()
        if SAVE_FIGURES:
            save_figure_protected(fig, COMPARISON_DIR / f"enzyme_pha_{OUTPUT_TAG}_figure{number}_{metric}_comparison.png", dpi=PNG_DPI)
        display(fig)
        plt.close(fig)
else:
    print("Comparison Figures 1–3 disabled.")


### Figure 4 — time within the configured minimum-distance cutoff

Uses Figure 3 CSVs only. A system is within when its **raw minimum heavy-atom PBC distance ≤ `DISTANCE_CUTOFF_A`**, default 3.5 Å. Integrate time with linear interpolation of threshold crossings between actual timestamps, then report `100 × within time / elapsed duration`. Rolling means are not used. Uneven spacing is supported; brief unobserved excursions cannot be recovered. Labels report percentage and ns. This is a descriptive proximity comparison, not a binding-affinity estimate. Saving the comparison PNG and CSV is controlled by `SAVE_WITHIN_COMPARISON`.


In [ ]:
if RUN_PLOTS:
    def duration_within_cutoff(time_ns, distance_A, cutoff_A):
        """Integrate time at or below cutoff, with linear interpolation between samples."""
        time_ns = np.asarray(time_ns, dtype=float)
        distance_A = np.asarray(distance_A, dtype=float)
        if (time_ns.ndim != 1 or distance_A.shape != time_ns.shape or len(time_ns) < 2
                or not np.isfinite(time_ns).all() or not np.isfinite(distance_A).all()
                or np.any(distance_A < 0) or not np.isfinite(cutoff_A) or cutoff_A <= 0):
            raise ValueError("Require at least two finite times/distances and a positive Å cutoff.")
        dt = np.diff(time_ns)
        if np.any(dt <= 0):
            raise ValueError("Sample times must be strictly increasing.")
        left, right = distance_A[:-1], distance_A[1:]
        inside_left, inside_right = left <= cutoff_A, right <= cutoff_A
        fraction = (inside_left & inside_right).astype(float)
        crossing = inside_left != inside_right
        crossing_position = (cutoff_A - left[crossing]) / (right[crossing] - left[crossing])
        fraction[crossing] = np.where(inside_left[crossing], crossing_position, 1 - crossing_position)
        return float(np.sum(dt * np.clip(fraction, 0, 1)))


    import hashlib
    within_rows = []
    for system in WITHIN_SYSTEMS:
        data, prefix = load_plot_data(system, "enzyme_pha_min_distance", "enzyme_pha_min_distance_A")
        if not np.allclose([data.time_ns.iloc[0], data.time_ns.iloc[-1]],
                           [START_NS, END_NS], atol=1e-8, rtol=0):
            raise ValueError(f"{system}: CSV must cover the entire common window; no extrapolation.")
        total_ns = float(data.time_ns.iloc[-1] - data.time_ns.iloc[0])
        within_ns = duration_within_cutoff(data.time_ns, data.enzyme_pha_min_distance_A, DISTANCE_CUTOFF_A)
        source_csv = Path(f"{prefix}_enzyme_pha_min_distance.csv")
        within_rows.append({
            "system": system.removesuffix("_gromacs"), "cutoff_A": DISTANCE_CUTOFF_A,
            "start_ns": float(data.time_ns.iloc[0]), "end_ns": float(data.time_ns.iloc[-1]),
            "within_time_ns": within_ns, "total_time_ns": total_ns,
            "percent_within": 100 * within_ns / total_ns, "sample_count": len(data),
            "min_sample_spacing_ns": float(data.time_ns.diff().dropna().min()),
            "max_sample_spacing_ns": float(data.time_ns.diff().dropna().max()),
            "method": "raw minimum distance <= cutoff; linear crossing interpolation",
            "source_csv": str(source_csv),
            "source_csv_sha256": hashlib.sha256(source_csv.read_bytes()).hexdigest(),
        })
    within_summary = pd.DataFrame(within_rows)
    display(within_summary[["system", "cutoff_A", "within_time_ns", "total_time_ns", "percent_within"]].round(3))

    # Edit the comparison plot here without rerunning any MD calculations.
    labels = [SYSTEM_LABELS[system] for system in WITHIN_SYSTEMS]
    colours = [SYSTEM_COLOURS[system] for system in WITHIN_SYSTEMS]
    fig, ax = plt.subplots(figsize=FIGSIZE)
    bars = ax.bar(labels, within_summary.percent_within, color=colours, width=0.65)
    for bar, row in zip(bars, within_summary.itertuples()):
        ax.annotate(f"{row.percent_within:.1f}%\n({row.within_time_ns:.1f} ns)",
                    (bar.get_x() + bar.get_width()/2, bar.get_height()),
                    xytext=(0, 5), textcoords="offset points", ha="center", va="bottom")
    ax.set(ylabel=f"Time with minimum distance ≤ {DISTANCE_CUTOFF_A:g} Å (%)",
           title=f"Enzyme–PHA proximity: {START_NS:g}–{END_NS:g} ns", ylim=(0, 115))
    ax.set_yticks(np.arange(0, 101, 20))
    ax.set_axisbelow(True)
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    if SAVE_WITHIN_COMPARISON:
        WITHIN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        cutoff_tag = f"{DISTANCE_CUTOFF_A:g}".replace(".", "p")
        within_prefix = WITHIN_OUTPUT_DIR / f"enzyme_pha_{OUTPUT_TAG}_within_{cutoff_tag}A_comparison"
        write_csv_protected(within_summary, f"{within_prefix}.csv")
        save_figure_protected(fig, f"{within_prefix}.png", dpi=PNG_DPI)
        print("Saved protected Figure 4 comparison outputs.")
    display(fig)
    plt.close(fig)
else:
    print("RUN_PLOTS=False; this section is disabled.")


## 6. Results and rerunning

Each system gets metric CSVs, an energy log and a summary of input files, selections, active-site checks and actual time coverage/spacing. After your first successful Run All, set `RUN_ANALYSIS=False` to redraw from these files. If you leave it True, complete compatible calculations are reused.

Changing sampling, selections or the time window requires a new `OUTPUT_TAG`. Cutoff/style changes can use the existing distance CSVs. A coarser stride can miss transient contacts; duration calculations use actual timestamps and interpolate threshold crossings.


## 7. Residue–PHA distances for Figures 5 and 6

Calculated automatically when `RUN_ANALYSIS=True`, after Figure 3's distances are saved. Compatible caches are reused; partial or incompatible results are protected.

For every sampled frame, each protein residue's heavy atoms are compared with all PHA heavy atoms using the original coordinates and same-frame periodic box. The minimum over all residues must reproduce Figure 3. Both 3.5 Å and 3.0 Å contacts are available from the same saved distances. All residues are retained and the verified ANC45 triad is marked.


In [ ]:
import hashlib
from matplotlib.colors import ListedColormap, BoundaryNorm

if len(RESIDUE_SYSTEMS) != 4 or len(set(RESIDUE_SYSTEMS)) != 4:
    raise ValueError("Figures 5 and 6 require four distinct configured systems.")
if not RESIDUE_CUTOFFS_A or any(not np.isfinite(c) or c <= 0 for c in RESIDUE_CUTOFFS_A):
    raise ValueError("Contact cutoffs must be positive finite Å values.")


def residue_paths(system):
    folder = BASE_DIR / system
    prefix = folder / "analysis" / f"{system.removesuffix('_gromacs')}_{OUTPUT_TAG}"
    return {"prefix": prefix, "tpr": folder / TPR_NAME,
            "xtc": folder / "analysis" / TRAJECTORY_NAME,
            "figure3": Path(f"{prefix}_enzyme_pha_min_distance.csv"),
            "figure3_summary": Path(f"{prefix}_summary.json"),
            "distances": Path(f"{prefix}_residue_pha_distances_A.csv"),
            "residues": Path(f"{prefix}_residue_pha_residues.csv"),
            "metadata": Path(f"{prefix}_residue_pha_metadata.json")}


def residue_input_stats(paths):
    return {key: [paths[key].stat().st_size, paths[key].stat().st_mtime_ns] for key in ("tpr", "xtc")}


def file_sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


def figure3_for_residues(system):
    paths = residue_paths(system)
    data = pd.read_csv(paths["figure3"])
    meta = json.loads(paths["figure3_summary"].read_text())
    columns = ["frame", "time_ns", "enzyme_pha_min_distance_A"]
    data = data[columns].copy()
    if len(data) < 2 or not np.isfinite(data.to_numpy()).all() or (data.enzyme_pha_min_distance_A < 0).any():
        raise ValueError(f"{system}: invalid Figure 3 data.")
    if not np.equal(data.frame, np.floor(data.frame)).all():
        raise ValueError("Figure 3 frame indices must be integers.")
    data["frame"] = data.frame.astype(int)
    if data.frame.iloc[0] != meta["reference_frame"] or not (np.diff(data.frame) == STRIDE).all() or np.any(np.diff(data.time_ns) <= 0):
        raise ValueError(f"{system}: Figure 3 sampling differs from STRIDE={STRIDE}.")
    if not np.allclose([data.time_ns.iloc[0], data.time_ns.iloc[-1]], [START_NS, END_NS], atol=1e-8, rtol=0):
        raise ValueError(f"{system}: Figure 3 does not cover the full comparison window.")
    if (meta["selections"]["protein"] != PROTEIN_SELECTION or meta["selections"]["pha"] != PHA_SELECTION
            or meta["selections"]["heavy"] != "TPR mass > 3 u" or meta["stride"] != STRIDE
            or meta["units"]["minimum_distance"] != "Å" or meta["units"]["time"] != "ns"):
        raise ValueError(f"{system}: incompatible Figure 3 selections, sampling or units.")
    stats = residue_input_stats(paths)
    for key in ("tpr", "xtc"):
        if paths[key].resolve() != Path(meta["inputs"][key]).resolve() or stats[key] != meta["input_size_mtime_ns"][key]:
            raise ValueError(f"{system}: {key} differs from the Figure 3 input.")
    return paths, data, meta


def load_residue_distances(system):
    paths, figure3, previous = figure3_for_residues(system)
    meta = json.loads(paths["metadata"].read_text())
    expected = {"protein": PROTEIN_SELECTION, "pha": PHA_SELECTION, "heavy": "TPR mass > 3 u"}
    if (meta["selections"] != expected or meta["stride"] != STRIDE
            or meta["window_ns"] != [START_NS, END_NS] or meta["units"] != {"time": "ns", "distance": "Å"}
            or meta["input_size_mtime_ns"] != residue_input_stats(paths)
            or meta["figure3_sha256"] != file_sha256(paths["figure3"])):
        raise ValueError(f"{system}: residue cache settings/provenance differ; calculate a new cache.")
    if meta["distance_csv_sha256"] != file_sha256(paths["distances"]) or meta["residue_csv_sha256"] != file_sha256(paths["residues"]):
        raise ValueError(f"{system}: a cached CSV changed after validation.")
    residues = pd.read_csv(paths["residues"], keep_default_na=False)
    data = pd.read_csv(paths["distances"])
    columns = residues.distance_column.tolist()
    if data.columns.tolist() != ["frame", "time_ns"] + columns or len(set(columns)) != len(columns):
        raise ValueError("Residue identifiers do not match distance columns.")
    if not np.array_equal(data.frame, figure3.frame) or not np.allclose(data.time_ns, figure3.time_ns, atol=1e-8, rtol=0):
        raise ValueError("Residue cache frames/times differ from Figure 3.")
    distances = data[columns].to_numpy(dtype=float)
    if not np.isfinite(distances).all() or np.any(distances < 0):
        raise ValueError("Invalid residue distances.")
    if not np.allclose(distances.min(axis=1), figure3.enzyme_pha_min_distance_A, atol=FIGURE3_TOLERANCE_A, rtol=0):
        raise ValueError("Residue minima do not reproduce Figure 3.")
    if residues.resid.duplicated().any() or not np.equal(residues.resid, np.floor(residues.resid)).all():
        raise ValueError("Plots require unique integer residue numbers; split chains if numbering overlaps.")
    return data, mark_active_site(residues), meta


### 7a. Calculate or reuse residue distances

This cell streams one trajectory frame at a time. It stores a small frame × residue distance matrix, not the full trajectory. Distance CSV columns identify each residue; the companion residue table records its original `resid`, `resname`, `segid`, topology `resindex`, heavy-atom count and atom names. The metadata JSON records exact frames/sampling, input fingerprints, selections, Å/ns units and all-frame Figure 3 validation. The first, middle and last analysed frames receive the source notebook's independent minimum-image check, which requires orthorhombic boxes and stops if that assumption fails.

In [ ]:
if RUN_RESIDUE_CALCULATION:
    from tempfile import TemporaryDirectory
    import MDAnalysis as mda
    from MDAnalysis.lib.distances import distance_array
    from MDAnalysis.lib.mdamath import triclinic_vectors

    # Complete validation of existing destinations before beginning any new calculation.
    pending = []
    for system in RESIDUE_SYSTEMS:
        paths, figure3, previous = figure3_for_residues(system)
        exists = [paths[k].exists() or paths[k].is_symlink() for k in ("distances", "residues", "metadata")]
        if any(exists) and not OVERWRITE_RESIDUE_DATA:
            if not all(exists):
                raise FileExistsError(f"{system}: incomplete residue cache; inspect it before explicitly enabling overwrite.")
            load_residue_distances(system)
            print(f"{system}: validated existing residue cache; reusing it.", flush=True)
        else:
            pending.append(system)

    for system in pending:
        paths, figure3, previous = figure3_for_residues(system)
        before = residue_input_stats(paths)
        figure3_hash = file_sha256(paths["figure3"])
        print(f"{system}: calculating {len(figure3)} sampled frames", flush=True)
        with TemporaryDirectory(prefix="ipha_residue_contacts_") as scratch:
            linked_xtc = Path(scratch) / "trajectory.xtc"
            linked_xtc.symlink_to(paths["xtc"])
            u = mda.Universe(str(paths["tpr"]), str(linked_xtc))
            try:
                protein = u.select_atoms(PROTEIN_SELECTION)
                validate_active_site(protein, system)
                pha = u.select_atoms(PHA_SELECTION)
                heavy = protein[protein.masses > 3]
                pha_heavy = pha[pha.masses > 3]
                if (len(u.atoms) != u.trajectory.n_atoms or len(protein.fragments) != 1
                        or not len(heavy) or not len(pha_heavy) or np.intersect1d(heavy.indices, pha_heavy.indices).size):
                    raise ValueError("Invalid topology/trajectory or overlapping/empty atom selections.")
                counts = {"system_atoms": len(u.atoms), "protein_atoms": len(protein),
                          "protein_residues": len(protein.residues), "protein_heavy_atoms": len(heavy),
                          "pha_atoms": len(pha), "pha_heavy_atoms": len(pha_heavy)}
                if any(value != previous["counts"][key] for key, value in counts.items()):
                    raise ValueError("Selection counts differ from Figure 3.")
                pha_identity = [{"resid": int(r.resid), "resname": str(r.resname), "segid": str(r.segid)} for r in pha.residues]
                if pha_identity != previous["counts"]["pha_identity"]:
                    raise ValueError("PHA residue identity differs from Figure 3.")
                for group in (heavy, pha_heavy):
                    if not np.isfinite(group.masses).all() or not np.isin(group.names.astype(str).astype('U1'), ['C','N','O','S']).all():
                        raise ValueError("Unexpected retained heavy-atom names/masses; inspect the topology.")
                resindices, starts = np.unique(heavy.resindices, return_index=True)
                if np.any(np.diff(heavy.resindices) < 0) or not np.array_equal(resindices, protein.residues.resindices):
                    raise ValueError("Every protein residue must have an ordered heavy-atom group.")
                residues = []
                for index in resindices:
                    residue = u.residues[index]
                    atoms = heavy[heavy.resindices == index]
                    residues.append({"distance_column": f"resindex_{index}_{residue.resname}{residue.resid}_A",
                                     "resindex": int(index), "resid": int(residue.resid),
                                     "resname": str(residue.resname), "segid": str(residue.segid),
                                     "heavy_atom_count": len(atoms), "heavy_atom_names": " ".join(atoms.names)})
                residues = mark_active_site(pd.DataFrame(residues))
                if residues.resid.duplicated().any():
                    raise ValueError("Multiple chains share residue numbers; a chain-specific plot is required.")
                values = np.empty((len(figure3), len(residues)), dtype=float)
                pair_buffer = np.empty((len(heavy), len(pha_heavy)), dtype=np.float64)
                frame_checks, max_error = [], 0.0
                for row_number, row in enumerate(figure3.itertuples(index=False)):
                    ts = u.trajectory[int(row.frame)]
                    if not np.isclose(ts.time / 1000, row.time_ns, atol=1e-8, rtol=0):
                        raise ValueError(f"Frame/time mismatch at {row.frame}.")
                    box = ts.dimensions.copy() if ts.dimensions is not None else None
                    if box is None or not np.isfinite(box).all() or np.any(box[:3] <= 0) or np.linalg.det(triclinic_vectors(box)) <= 0:
                        raise ValueError(f"Invalid box at frame {row.frame}.")
                    enzyme_xyz, pha_xyz = heavy.positions.copy(), pha_heavy.positions.copy()
                    if not np.isfinite(enzyme_xyz).all() or not np.isfinite(pha_xyz).all():
                        raise ValueError("Non-finite coordinates.")
                    # Original coordinates + same-frame box, with no RMSD fitting or transformations.
                    distance_array(enzyme_xyz, pha_xyz, box=box, result=pair_buffer)
                    per_atom_minimum = pair_buffer.min(axis=1)
                    values[row_number] = np.minimum.reduceat(per_atom_minimum, starts)
                    error = abs(float(values[row_number].min()) - row.enzyme_pha_min_distance_A)
                    max_error = max(max_error, error)
                    if error > FIGURE3_TOLERANCE_A:
                        raise ValueError(f"Residue minima differ from Figure 3 at frame {row.frame}: {error} Å.")
                    if not np.array_equal(heavy.positions, enzyme_xyz) or not np.array_equal(ts.dimensions, box):
                        raise AssertionError("Coordinates or box changed during distance calculation.")
                    if row_number in {0, len(figure3)//2, len(figure3)-1}:
                        if not np.allclose(box[3:], 90, atol=1e-4):
                            raise ValueError("Independent check requires orthorhombic boxes; inspect any non-orthorhombic input before adapting this check.")
                        delta = enzyme_xyz.astype(float)[:, None, :] - pha_xyz.astype(float)[None, :, :]
                        delta -= box[:3] * np.rint(delta / box[:3])
                        independent = np.minimum.reduceat(np.linalg.norm(delta, axis=2).min(axis=1), starts)
                        check_error = float(np.max(np.abs(independent - values[row_number])))
                        assert check_error < 2e-5
                        frame_checks.append({"frame": int(row.frame), "time_ns": float(row.time_ns),
                                             "max_residue_error_A": check_error})
                    if row_number % 2000 == 0:
                        print(f"  {row_number + 1}/{len(figure3)} frames; Figure 3 max error {max_error:.3g} Å", flush=True)
                atom_ids = {"protein_heavy_atom_indices": heavy.indices.tolist(), "pha_heavy_atom_indices": pha_heavy.indices.tolist()}
            finally:
                u.trajectory.close()
        if residue_input_stats(paths) != before or file_sha256(paths["figure3"]) != figure3_hash:
            raise RuntimeError("A source file changed during calculation.")
        result = pd.DataFrame(values, columns=residues.distance_column)
        result.insert(0, "time_ns", figure3.time_ns.to_numpy())
        result.insert(0, "frame", figure3.frame.to_numpy())
        result.to_csv(paths["distances"], index=False, float_format="%.8f")
        residues.to_csv(paths["residues"], index=False)
        steps = np.diff(figure3.time_ns)
        metadata = {"system": system, "active_site": previous["active_site"], "inputs": {k: str(paths[k]) for k in ("tpr", "xtc")},
                    "input_size_mtime_ns": before, "figure3_csv": str(paths["figure3"]),
                    "figure3_sha256": figure3_hash, "selections": {"protein": PROTEIN_SELECTION,
                    "pha": PHA_SELECTION, "heavy": "TPR mass > 3 u"}, "counts": counts,
                    "pha_identity": pha_identity, **atom_ids, "units": {"time": "ns", "distance": "Å"},
                    "stride": STRIDE, "window_ns": [START_NS, END_NS], "sample_count": len(result),
                    "frame_start": int(figure3.frame.iloc[0]), "frame_end": int(figure3.frame.iloc[-1]),
                    "sampling_interval_ns": {"min": float(steps.min()), "max": float(steps.max()),
                                             "uniform": bool(np.allclose(steps, steps[0], atol=1e-8, rtol=0))},
                    "pbc": "minimum image with each frame's original box; unaligned coordinates",
                    "distance_definition": "minimum over residue heavy atoms and all PHA heavy atoms",
                    "figure3_validation_frames": len(result), "figure3_max_abs_error_A": max_error,
                    "independent_checks": frame_checks, "MDAnalysis_version": mda.__version__,
                    "distance_csv_sha256": file_sha256(paths["distances"]),
                    "residue_csv_sha256": file_sha256(paths["residues"])}
        paths["metadata"].write_text(json.dumps(metadata, indent=2) + "\n")
        print(f"Saved residue distances, identifiers and settings: {paths['prefix']}_residue_pha_*", flush=True)
        del result, values
else:
    print("Residue trajectory calculation skipped. Plotting below reads the saved residue CSVs.")


### 7b. Load saved data and apply Figure 4's time convention

No MDAnalysis or trajectory reading is required from this point. All four systems must span the entire common window. Occupancy is **100 × estimated time at or below cutoff / 1000 ns**, including every period when the polymer is away. Each residue is classified once per sample, regardless of the number of atom pairs. Crossing times are linearly interpolated between neighbouring raw distances, exactly as in Figure 4; no rolling means enter contact calculations. The integration is weighted by the actual timestamp differences and also works for uneven sampling.

The binary map in Figure 6 shows the classification **at the saved times**. Time cells extend to adjacent time midpoints (clipped to the configured time window); this visual binning does not replace the interpolated occupancy calculation. White means no contact. Original residue numbers are used, with missing residue numbers masked grey rather than renumbered. All residues and all sampled times remain present.

In [ ]:
def residue_duration_within_cutoff(time_ns, distance_A, cutoff_A):
    """Integrate time at or below cutoff, with linear interpolation between samples."""
    time_ns = np.asarray(time_ns, dtype=float)
    distance_A = np.asarray(distance_A, dtype=float)
    if (time_ns.ndim != 1 or distance_A.shape != time_ns.shape or len(time_ns) < 2
            or not np.isfinite(time_ns).all() or not np.isfinite(distance_A).all()
            or np.any(distance_A < 0) or not np.isfinite(cutoff_A) or cutoff_A <= 0):
        raise ValueError("Require at least two finite times/distances and a positive Å cutoff.")
    dt = np.diff(time_ns)
    if np.any(dt <= 0):
        raise ValueError("Sample times must be strictly increasing.")
    left, right = distance_A[:-1], distance_A[1:]
    inside_left, inside_right = left <= cutoff_A, right <= cutoff_A
    fraction = (inside_left & inside_right).astype(float)
    crossing = inside_left != inside_right
    crossing_position = (cutoff_A - left[crossing]) / (right[crossing] - left[crossing])
    fraction[crossing] = np.where(inside_left[crossing], crossing_position, 1 - crossing_position)
    return float(np.sum(dt * np.clip(fraction, 0, 1)))


# Small analytic checks include unequal time intervals and crossing at the cutoff.
assert np.isclose(residue_duration_within_cutoff([0, 3, 10], [2, 2, 5], 3.5), 6.5)
assert residue_duration_within_cutoff([0, 10], [3.5, 3.5], 3.5) == 10
assert residue_duration_within_cutoff([0, 10], [5, 5], 3.5) == 0

residue_cache = {}
if RUN_PLOTS or RUN_PERIOD_TABLES or EXPORT_REPRESENTATIVE_FRAMES:
    for system in RESIDUE_SYSTEMS:
        data, residues, metadata = load_residue_distances(system)
        residue_cache[system] = (data, residues, metadata)
        print(f"{system}: {len(residues)} residues; {len(data)} frames; "
              f"{data.time_ns.iloc[0]:g}–{data.time_ns.iloc[-1]:g} ns; "
              f"sample spacing {metadata['sampling_interval_ns']['min']:g}–{metadata['sampling_interval_ns']['max']:g} ns")


def residue_cutoff_tag(cutoff):
    return f"{cutoff:.1f}".replace(".", "p")


def residue_panel_title(system):
    return SYSTEM_LABELS[system]


## 8. Figure 5 — Residue–PHA Contact Occupancy

Figure 5 uses **3.5 Å** (`FIGURE5_CUTOFF_A`). Its four panels retain the full residue range and 0–100% axes. P3HO₄ is blue and P3HO₈ orange. Red overlays/labels identify **SER138, ASP194 and HSD226**, including zero-occupancy active-site residues. CSVs include all residues and an `active_site` column. Printed tables show the top five and all active-site members, with active-site rows highlighted in red. Occupancy uses elapsed time and linear threshold-crossing interpolation, including separated periods.


In [ ]:
if RUN_PLOTS:
    for cutoff in (FIGURE5_CUTOFF_A,):
        tag = residue_cutoff_tag(cutoff)
        fig, axes = plt.subplots(2, 2, figsize=PANEL_FIGSIZE, sharex=True, sharey=True)
        tables = []
        residue_min = min(int(residues.resid.min()) for _, residues, _ in residue_cache.values())
        residue_max = max(int(residues.resid.max()) for _, residues, _ in residue_cache.values())
        for ax, system in zip(axes.flat, RESIDUE_SYSTEMS):
            data, residues, metadata = residue_cache[system]
            table = residues.copy()
            table.insert(0, "system", system.removesuffix("_gromacs"))
            table["cutoff_A"] = cutoff
            table["within_time_ns"] = [residue_duration_within_cutoff(data.time_ns, data[column], cutoff)
                                       for column in residues.distance_column]
            table["total_time_ns"] = float(data.time_ns.iloc[-1] - data.time_ns.iloc[0])
            table["percent_within"] = 100 * table.within_time_ns / table.total_time_ns
            table["method"] = "distance <= cutoff; linear interpolation of crossings; full elapsed duration"
            table["source_distances"] = str(residue_paths(system)["distances"])
            tables.append(table)
            colour = SYSTEM_COLOURS[system]
            ax.bar(table.resid, table.percent_within, color=colour, width=0.85)
            active = table.loc[table.active_site]
            ax.scatter(active.resid, active.percent_within, color=ACTIVE_SITE_COLOUR,
                       marker="D", s=22, zorder=4, label="ANC45 active site")
            for row in active.itertuples():
                ax.annotate(f"{row.resname}{row.resid}", (row.resid, row.percent_within),
                            xytext=(0, -6 if row.percent_within > 85 else 6), textcoords="offset points", rotation=90,
                            ha="center", va="top" if row.percent_within > 85 else "bottom", color=ACTIVE_SITE_COLOUR, fontsize=8)
            ax.legend(fontsize=8)
            ax.set(title=residue_panel_title(system), xlabel="Enzyme residue number",
                   ylabel="Analysed time within cutoff (%)", ylim=(0, 100), xlim=(residue_min-1, residue_max+1))
            ax.set_axisbelow(True)
            ax.grid(axis="y", alpha=0.3)
            print(f"{system}, ≤ {cutoff:g} Å — top residues:")
            shown = pd.concat([table.nlargest(5, "percent_within"), table.loc[table.active_site]]).drop_duplicates("resindex")
            display(highlight_table(shown[["resid", "resname", "active_site", "within_time_ns", "percent_within"]].round(3)))
            if SAVE_RESIDUE_FIGURES:
                write_csv_protected(table, f"{residue_paths(system)['prefix']}_residue_pha_occupancy_{tag}A.csv")
        fig.suptitle(f"Residue–PHA Contact Occupancy — distance ≤ {cutoff:.1f} Å ({START_NS:g}–{END_NS:g} ns)")
        fig.tight_layout()
        if SAVE_RESIDUE_FIGURES:
            RESIDUE_COMPARISON_DIR.mkdir(parents=True, exist_ok=True)
            prefix = RESIDUE_COMPARISON_DIR / f"enzyme_pha_{OUTPUT_TAG}_figure5_residue_occupancy_{tag}A"
            write_csv_protected(pd.concat(tables, ignore_index=True), f"{prefix}.csv")
            save_figure_protected(fig, f"{prefix}.png", dpi=PNG_DPI)
            print("Saved protected Figure 5 comparison outputs.")
        display(fig)
        plt.close(fig)
else:
    print("RUN_PLOTS=False; this section is disabled.")


## 9. Figure 6 — Enzyme–PHA Contacts Over Time

Binary maps retain all residues and contact-free periods: white = no contact, blue/orange = contact for P3HO₄/P3HO₈ respectively, grey = an absent residue number. Each panel has its own colourbar. Red dashed lines and labelled residue ticks locate the verified ANC45 active site. Contacts are unsmoothed and use actual sample-time cells bounded by neighbouring midpoints. Per-system supporting CSVs contain frame/time and full 0/1 residue contacts; the shared manifest identifies residue tables and measured sampling. The map displays discrete sampled states; occupancy tables use interpolated cutoff crossings.


In [ ]:
if RUN_PLOTS:
    for cutoff in RESIDUE_CUTOFFS_A:
        tag = residue_cutoff_tag(cutoff)
        fig, axes = plt.subplots(2, 2, figsize=PANEL_FIGSIZE, sharex=True, sharey=True, layout="constrained")
        manifest = []
        residue_min = min(int(residues.resid.min()) for _, residues, _ in residue_cache.values())
        residue_max = max(int(residues.resid.max()) for _, residues, _ in residue_cache.values())
        residue_numbers = np.arange(residue_min, residue_max + 1)
        residue_edges = np.arange(residue_min - 0.5, residue_max + 1.5)
        for ax, system in zip(axes.flat, RESIDUE_SYSTEMS):
            data, residues, metadata = residue_cache[system]
            times = data.time_ns.to_numpy()
            time_edges = np.r_[START_NS, (times[:-1] + times[1:]) / 2, END_NS]
            contacts = (data[residues.distance_column].to_numpy() <= cutoff).astype(np.uint8)
            grid = np.ma.masked_all((len(residue_numbers), len(times)), dtype=np.uint8)
            grid[residues.resid.to_numpy(dtype=int) - residue_min, :] = contacts.T
            contact_cmap = ListedColormap(["white", SYSTEM_COLOURS[system]])
            contact_cmap.set_bad("0.85")
            contact_norm = BoundaryNorm([-0.5, 0.5, 1.5], contact_cmap.N)
            mesh = ax.pcolormesh(time_edges, residue_edges, grid, cmap=contact_cmap,
                                 norm=contact_norm, shading="flat", rasterized=True)
            ax.set(title=residue_panel_title(system), xlabel="Time (ns)", ylabel="Enzyme residue number",
                   xlim=(START_NS, END_NS), ylim=(residue_min-0.5, residue_max+0.5))
            ticks = sorted(set(np.linspace(residue_min, residue_max, 6, dtype=int)) | set(ACTIVE_SITE_RESIDUES))
            ax.set_yticks(ticks, [f"{ACTIVE_SITE_RESIDUES[r]}{r}" if r in ACTIVE_SITE_RESIDUES else str(r) for r in ticks])
            ax.tick_params(labelleft=True)
            for tick, resid in zip(ax.get_yticklabels(), ticks):
                if resid in ACTIVE_SITE_RESIDUES:
                    tick.set_color(ACTIVE_SITE_COLOUR)
                    tick.set_fontweight("bold")
            for resid in ACTIVE_SITE_RESIDUES:
                ax.axhline(resid, color=ACTIVE_SITE_COLOUR, linestyle="--", linewidth=0.6, alpha=0.7)
            colourbar = fig.colorbar(mesh, ax=ax, ticks=[0, 1], fraction=0.025, pad=0.02)
            colourbar.ax.set_yticklabels(["No contact", "Contact"])
            contact_path = Path(f"{residue_paths(system)['prefix']}_residue_pha_contacts_{tag}A.csv")
            if SAVE_RESIDUE_FIGURES:
                binary = pd.DataFrame(contacts, columns=residues.distance_column.str.removesuffix("_A") + "_contact")
                binary.insert(0, "time_ns", times)
                binary.insert(0, "frame", data.frame.to_numpy())
                write_csv_protected(binary, contact_path)
            manifest.append({"system": system, "cutoff_A": cutoff, "contact_csv": str(contact_path),
                             "residue_identifiers_csv": str(residue_paths(system)["residues"]),
                             "distance_csv": str(residue_paths(system)["distances"]),
                             "frames": len(data), "residues": len(residues), "start_ns": times[0], "end_ns": times[-1],
                             "resid_min": int(residues.resid.min()), "resid_max": int(residues.resid.max()),
                             "sample_spacing_min_ns": float(np.diff(times).min()),
                             "sample_spacing_max_ns": float(np.diff(times).max()),
                             "no_contact_sample_count": int((contacts.sum(axis=1) == 0).sum()),
                             "definition": "1 if residue minimum heavy-atom PBC distance <= cutoff, otherwise 0",
                             "map_time_cells": "midpoints between actual sample timestamps, clipped to full window"})
        fig.suptitle(f"Enzyme–PHA Contacts Over Time — distance ≤ {cutoff:.1f} Å")
        if SAVE_RESIDUE_FIGURES:
            RESIDUE_COMPARISON_DIR.mkdir(parents=True, exist_ok=True)
            prefix = RESIDUE_COMPARISON_DIR / f"enzyme_pha_{OUTPUT_TAG}_figure6_contacts_over_time_{tag}A"
            write_csv_protected(pd.DataFrame(manifest), f"{prefix}.csv")
            save_figure_protected(fig, f"{prefix}.png", dpi=PNG_DPI)
            print("Saved protected Figure 6 comparison outputs and four binary-table indexes.")
        display(fig)
        plt.close(fig)
else:
    print("RUN_PLOTS=False; this section is disabled.")


### Saved data

Per-system files include the residue-distance CSV, residue identities and validation metadata, plus occupancy and binary-contact tables. Figures 4–6 and four-system comparison panels save in the shared `PHA_new_Enzyme/analysis/` folder.

`RUN_ANALYSIS=False` redraws from CSVs. TPR/XTC files remain available for provenance checks but are not decoded for redraws. Existing different CSVs are protected, and figure revisions get unique filenames.


## 10. Leading residues by period and representative PDB snapshots

Outputs per system: **top ten overall**, **top ten in each configured period** (default 200 ns), **complete all-residue rankings**, and **one standalone snapshot per period** (five by default). An asterisk (`*`) marks contact percentages strictly greater than 10% in both ranking tables (using unrounded values). These are residues with the highest PHA contact occupancy, not assignments of catalytic function, hydrogen bonding or binding strength.

Run All automatically creates and saves ranking tables and representative-frame choices from the saved distance CSVs. **PDB extraction only runs if `EXPORT_REPRESENTATIVE_FRAMES=True`** in the first settings cell. It reads the chosen frame per configured period and validates each PDB against the cached contacts. Actual saved sampling is measured rather than assumed. Contacts use protein / LIG heavy atoms, each original periodic box and the configured cutoff (default 3.5 Å).

Occupancy uses the existing **actual-timestamp weighting and linear interpolation of cutoff crossings**. The denominator is the configured elapsed duration (1000 ns overall and 200 ns per period by default), including separation. Boundary distances are interpolated if necessary. Adjacent integrations share only a zero-duration boundary; sample choices use `[start, end)`, with END_NS included in the final period. Contacts are counted once per residue, and zero-contact residues do not fill empty ranks. Exact occupancy ties use ascending topology `resindex`.

**Snapshot rule:** estimate the duration of each exact contact pattern across **all enzyme residues**, splitting at the same interpolated cutoff crossings. First choose the majority state: no contact if it occupies at least half the window, otherwise contact. Within that state, choose the most prevalent pattern represented by an actual sampled frame, then the sample nearest the window midpoint. Duration/midpoint ties within 10⁻⁹ ns favour the earlier time and saved frame index. This selects a typical pattern without maximising contact count or minimising distance. The most prevalent contact pattern can still occupy a small fraction of a varied window; its percentage is recorded in the snapshot CSV. Interpolated patterns are estimates, not direct observations between samples.

The snapshot table lists contacting members of that period's top five. If contacts involve only other residues, those are explicitly identified instead. **“No contact within 3.5 Å” means no enzyme residue meets the cutoff**, not merely that the top five are absent. All listed contacts are checked in the saved data and exported PDB.

Every ranking CSV includes `active_site`; additional complete all-residue rankings preserve zero-contact residues and triad members below the top ten. Active-site rows are highlighted in displayed tables, and snapshot selections list contacting active-site residues separately. With optional PDB export disabled, only the tables and frame choices are saved.


In [ ]:
def period_data(data, residues, start, end):
    """Clip the piecewise-linear residue distances, retaining zero-duration boundaries."""
    times = data.time_ns.to_numpy()
    values = data[residues.distance_column].to_numpy()
    inner = (times > start) & (times < end)
    clipped_times = np.r_[start, times[inner], end]
    clipped = np.vstack(([np.interp(start,times,column) for column in values.T],
                         values[inner], [np.interp(end,times,column) for column in values.T]))
    durations = np.array([residue_duration_within_cutoff(clipped_times,column,CONTACT_CUTOFF_A)
                          for column in clipped.T])
    return clipped_times, clipped, durations


def top_ten(residues, durations, duration):
    ranked = residues.assign(contact_time_ns=durations, occupancy_percent=100*durations/duration)
    ranked = ranked.loc[ranked.contact_time_ns > 0].sort_values(
        ["occupancy_percent","resindex"], ascending=[False,True]).head(10).copy()
    ranked.insert(0,"rank",np.arange(1,len(ranked)+1))
    return ranked


def contact_percentage_text(value):
    return f"{value:.2f}%" + (" *" if value > 10.0 else "")


def contact_pattern_durations(times, values):
    """Joint binary patterns integrated over the same linear cutoff crossings."""
    totals = {}
    for t0,t1,left,right in zip(times[:-1],times[1:],values[:-1],values[1:]):
        inside_left,inside_right = left <= CONTACT_CUTOFF_A, right <= CONTACT_CUTOFF_A
        changing = inside_left != inside_right
        fractions = (CONTACT_CUTOFF_A-left[changing])/(right[changing]-left[changing])
        edges = np.unique(np.r_[0.0,np.clip(fractions,0,1),1.0])
        for a,b in zip(edges[:-1],edges[1:]):
            middle = left + ((a+b)/2)*(right-left)
            pattern = tuple(np.flatnonzero(middle <= CONTACT_CUTOFF_A))
            totals[pattern] = totals.get(pattern,0.0) + (b-a)*(t1-t0)
    assert np.isclose(sum(totals.values()),times[-1]-times[0],atol=1e-8,rtol=0)
    return totals


def representative_sample(data, residues, start, end, times, values, durations):
    pattern_times = contact_pattern_durations(times,values)
    # Independent agreement of the joint-pattern integral and existing per-residue method.
    reconstructed = np.zeros(len(residues))
    for pattern,duration in pattern_times.items():
        reconstructed[list(pattern)] += duration
    assert np.allclose(reconstructed,durations,atol=1e-8,rtol=0)
    separated_time = pattern_times.get((),0.0)
    separated = separated_time >= (end-start)/2
    sample_times = data.time_ns.to_numpy()
    positions = np.flatnonzero((sample_times >= start) &
                              ((sample_times < end) | ((end == END_NS) & (sample_times == end))))
    contacts = data[residues.distance_column].to_numpy() <= CONTACT_CUTOFF_A
    patterns = {int(pos):tuple(np.flatnonzero(contacts[pos])) for pos in positions}
    eligible = [pos for pos,pattern in patterns.items() if (len(pattern)==0)==separated]
    if not eligible:
        raise ValueError("The estimated majority state has no actual sample; cannot export an unobserved frame.")
    best_time = max(pattern_times.get(patterns[pos],0.0) for pos in eligible)
    candidates = [pos for pos in eligible if np.isclose(pattern_times.get(patterns[pos],0.0),best_time,
                                                       atol=SNAPSHOT_TIE_NS,rtol=0)]
    nearest = min(abs(sample_times[pos]-(start+end)/2) for pos in candidates)
    tied = [pos for pos in candidates if np.isclose(abs(sample_times[pos]-(start+end)/2),nearest,
                                                   atol=SNAPSHOT_TIE_NS,rtol=0)]
    pos = min(tied,key=lambda j:(sample_times[j],int(data.frame.iloc[j])))
    return pos, separated, 100*separated_time/(end-start), 100*best_time/(end-start)


def residue_text(row):
    return f"{row.resname}{int(row.resid)}"


def vmd_highlight(rows):
    # Snapshots explicitly assign chain A to the enzyme; names/numbers are verified on re-read.
    terms = [f"(resid {int(row.resid)} and resname {row.resname})" for row in rows.itertuples()]
    return "chain A and (" + " or ".join(terms) + ")" if terms else "none"


# Uneven timestamps and a shared boundary: interval occupancies must add exactly.
assert np.isclose(residue_duration_within_cutoff([0,3,10],[2,2,5],3.5),6.5)
if CONTACT_CUTOFF_A == 3.5:
    assert contact_pattern_durations(np.array([0.,1.,2.]),np.array([[2.],[5.],[2.]])) == {(0,):1.0,():1.0}


In [ ]:
if RUN_PERIOD_TABLES or EXPORT_REPRESENTATIVE_FRAMES:
    # Build all three outputs from the saved data before opening any trajectory.
    period_results = {}
    for system in RESIDUE_SYSTEMS:
        data,residues,metadata = load_residue_distances(system)
        assert residues.resid.is_unique and residues.resindex.is_unique
        times = data.time_ns.to_numpy()
        print(system, "saved timestamp coverage/spacing:", spacing_report(times))
        _,_,overall_durations = period_data(data,residues,START_NS,END_NS)
        overall = top_ten(residues,overall_durations,END_NS-START_NS)
        all_overall = residues.assign(contact_time_ns=overall_durations,
                                       occupancy_percent=100 * overall_durations / (END_NS - START_NS))
        all_overall = all_overall.sort_values(["occupancy_percent", "resindex"], ascending=[False, True]).copy()
        all_overall.insert(0, "rank", np.arange(1, len(all_overall) + 1))
        all_periods = []
        periods,snapshots,window_durations = [],[],[]
        for start,end in SUMMARY_WINDOWS_NS:
            clipped_times,clipped,durations = period_data(data,residues,start,end)
            window_durations.append(durations)
            all_ranked = residues.assign(contact_time_ns=durations, occupancy_percent=100 * durations / (end - start))
            all_ranked = all_ranked.sort_values(["occupancy_percent", "resindex"], ascending=[False, True]).copy()
            all_ranked.insert(0, "rank", np.arange(1, len(all_ranked) + 1))
            all_ranked["window_start_ns"], all_ranked["window_end_ns"] = start, end
            all_ranked["time_window"] = f"{start:g}–{end:g} ns"
            all_periods.append(all_ranked)
            ranked = top_ten(residues,durations,end-start)
            label = f"{start:g}–{end:g} ns"
            ranked.insert(0,"time_window",label)
            ranked["window_start_ns"],ranked["window_end_ns"] = start,end
            periods.append(ranked)
            pos,separated,separated_percent,pattern_percent = representative_sample(
                data,residues,start,end,clipped_times,clipped,durations)
            selected_time,frame = float(data.time_ns.iloc[pos]),int(data.frame.iloc[pos])
            actual = data.iloc[pos][residues.distance_column].to_numpy(dtype=float) <= CONTACT_CUTOFF_A
            contacting = residues.loc[actual]
            # Snapshot annotations retain the original five leading residues.
            snapshot_leaders = ranked.head(5)
            listed = snapshot_leaders.loc[snapshot_leaders.distance_column.isin(contacting.distance_column)]
            text = "; ".join(residue_text(row) for row in listed.itertuples())
            if contacting.empty:
                text = f"no contact within {CONTACT_CUTOFF_A:g} Å"
            elif listed.empty:
                listed = contacting
                text = "Other residues (none of this period's top five): " + "; ".join(
                    residue_text(row) for row in listed.itertuples())
            time_tag = np.format_float_positional(selected_time,trim="-").replace(".","p")
            filename = f"{system.removesuffix('_gromacs')}_{OUTPUT_TAG}_snapshot_{time_tag}ns.pdb"
            snapshots.append({"time_window":label,"window_start_ns":start,"window_end_ns":end,
                              "time_ns":selected_time,"saved_frame_index":frame,
                              "state":"separated" if separated else "contact",
                              "contacting_leading_residues":text,"snapshot_filename":filename,
                              "vmd_selection":vmd_highlight(listed),
                              "contacting_active_site_residues":"; ".join(residue_text(r) for r in contacting.loc[contacting.active_site].itertuples()),
                              "active_site_vmd_selection":vmd_highlight(residues.loc[residues.active_site]),
                              "all_contacting_residues":"; ".join(residue_text(row) for row in contacting.itertuples()),
                              "separated_time_percent":separated_percent,"selected_pattern_percent":pattern_percent})
        assert np.allclose(np.sum(window_durations,axis=0),overall_durations,atol=1e-8,rtol=0)
        period_results[system] = {"data":data,"residues":residues,"metadata":metadata,
                                  "overall":overall,"periods":pd.concat(periods,ignore_index=True),
                                  "all_overall":all_overall,"all_periods":pd.concat(all_periods,ignore_index=True),
                                  "snapshots":pd.DataFrame(snapshots)}
        print(f"{system}: measured saved spacing; occupancy checks passed; "
              f"{sum(row['state']=='separated' for row in snapshots)}/{len(SUMMARY_WINDOWS_NS)} windows are separation-dominated.")
else:
    print("RUN_PERIOD_TABLES or EXPORT_REPRESENTATIVE_FRAMES=False; this section is disabled.")


### 10a. Export only the selected frames per system

Use **`step6.2_npt.gro`** as the starting structure. Before extraction, compare its full atom count and ordered atom names, residue names and residue numbers with `step7_production.tpr`, check the XTC atom count, and verify the cached source fingerprints and the processing record's full-`System` output. XTC has no atom identities: ordering is established through this matching topology and recorded full-system provenance, not atom count alone. The GRO coordinates are not used as snapshot coordinates.

Production bonds make the enzyme and PHA individually whole. PHA is translated only by an **integer periodic lattice vector** to the image containing the closest enzyme–PHA heavy-atom pair. There is no fitting, docking, or arbitrary translation towards the enzyme: the original minimum-image physical separation is retained, including separated snapshots. Exported nonperiodic contact classifications must match every cached residue classification; if one whole PHA image cannot reproduce them, export stops rather than fabricating contacts.

PDBs include the complete enzyme and PHA, including hydrogens, without solvent or ions. Original residue names, numbers and atom names are retained. GRO has no chain identifiers; snapshots explicitly use **chain A / segment PROA** for the enzyme and **chain B / segment LIG** for PHA, mapping the cached `seg_0_PROA` / `seg_1_LIG`. Selections below use these verified PDB identifiers. Open each PDB directly in VMD (one coordinate frame); no trajectory-frame conversion is required. Display PHA with `chain B and resname LIG`.

Re-read each staged PDB and check identities, bonds, all residue contact classifications, and minimum separation against its saved frame. PDB coordinates have 0.001 Å precision; distance differences up to 0.002 Å are accepted, but **no cutoff classification may change**. Only these selected-frame/export validation distances are calculated. Existing different result files are protected, and all validation completes before the final save.

In [ ]:
if EXPORT_REPRESENTATIVE_FRAMES:
    import MDAnalysis as mda
    from MDAnalysis.lib.distances import distance_array, minimize_vectors
    from MDAnalysis.lib.mdamath import make_whole, triclinic_vectors
    from tempfile import TemporaryDirectory


    def verify_snapshot_inputs(system, result):
        paths = residue_paths(system)
        structure = system_inputs(system)["gro"]
        validate_topology_reference(system)
        topology = mda.Universe(str(paths["tpr"]))
        gro = mda.Universe(str(structure))
        try:
            assert len(gro.atoms) == len(topology.atoms) == result["metadata"]["counts"]["system_atoms"]
            for attribute in ("names","resnames","resids"):
                if not np.array_equal(getattr(gro.atoms,attribute),getattr(topology.atoms,attribute)):
                    raise ValueError(f"{system}: step6.2_npt.gro / TPR atom ordering differs ({attribute}).")
        finally:
            gro.trajectory.close()
        settings = {}
        for line in (paths["xtc"].parent / PROCESSING_RECORD_NAME).read_text().splitlines():
            if line.strip() and not line.lstrip().startswith("#"):
                key,value = line.split("=",1)
                settings[key.strip()] = value.strip()
        assert settings["OUTPUT_GROUP"] == "System"
        assert (paths["xtc"].parent/settings["TPR"]).resolve() == paths["tpr"].resolve()
        assert (paths["xtc"].parent/settings["PROCESSED"]).resolve() == paths["xtc"].resolve()
        # Confirm that System in the processing index is the full atom order.
        system_indices,inside = [],False
        for line in (paths["xtc"].parent/settings["INDEX"]).read_text().splitlines():
            if line.strip().startswith("["):
                inside = line.strip().strip("[]").strip() == "System"
            elif inside:
                system_indices.extend(int(value) for value in line.split())
        assert np.array_equal(system_indices,np.arange(1,len(topology.atoms)+1))
        assert residue_input_stats(paths) == result["metadata"]["input_size_mtime_ns"]
        return topology,structure


    def stage_system_snapshots(system, result, scratch):
        """Extract/validate only the selected samples; return immutable PDB payloads."""
        paths = residue_paths(system)
        u,structure = verify_snapshot_inputs(system,result)
        # The disposable symlink keeps XTC offset-index files out of the source folders.
        linked = scratch / f"{system}.xtc"
        linked.symlink_to(paths["xtc"])
        u.load_new(str(linked))
        payloads,validation = {},[]
        try:
            assert u.trajectory.n_atoms == len(u.atoms)
            protein,pha = u.select_atoms(PROTEIN_SELECTION),u.select_atoms(PHA_SELECTION)
            assert len(protein.fragments) == len(pha.fragments) == 1
            assert protein.fragments[0] == protein and pha.fragments[0] == pha
            heavy,ligand_heavy = protein[protein.masses > 3],pha[pha.masses > 3]
            meta,residues = result["metadata"],result["residues"]
            assert np.array_equal(heavy.indices,meta["protein_heavy_atom_indices"])
            assert np.array_equal(ligand_heavy.indices,meta["pha_heavy_atom_indices"])
            assert np.array_equal(heavy.residues.resindices,residues.resindex)
            assert np.array_equal(heavy.residues.resids,residues.resid)
            assert np.array_equal(heavy.residues.resnames,residues.resname)
            assert np.array_equal(heavy.residues.segids,residues.segid)
            assert set(protein.segids) == {"seg_0_PROA"} and set(pha.segids) == {"seg_1_LIG"}
            starts = np.r_[0,np.cumsum(residues.heavy_atom_count.to_numpy(dtype=int))[:-1]]
            selected = protein + pha
            snapshot = mda.Merge(protein,pha)
            snapshot.segments[0].segid = "PROA"
            snapshot.segments[1].segid = "LIG"
            snapshot.add_TopologyAttr("chainIDs",["A"]*len(protein)+["B"]*len(pha))
            snapshot.add_TopologyAttr("record_types",["ATOM"]*len(protein)+["HETATM"]*len(pha))
            snapshot.add_TopologyAttr("altLocs",[""]*len(selected))
            snapshot.add_TopologyAttr("icodes",[""]*len(snapshot.residues))
            snapshot.add_TopologyAttr("occupancies",np.ones(len(selected)))
            snapshot.add_TopologyAttr("tempfactors",np.zeros(len(selected)))
            snapshot.add_TopologyAttr("formalcharges",np.zeros(len(selected),dtype=int))
            protein_heavy_local = np.flatnonzero(protein.masses > 3)
            pha_heavy_local = np.flatnonzero(pha.masses > 3)
            bond_indices = snapshot.bonds.indices
            saved = result["data"].set_index("frame")
            for row in result["snapshots"].itertuples(index=False):
                ts = u.trajectory[int(row.saved_frame_index)]
                assert np.isclose(ts.time/1000,row.time_ns,atol=1e-8,rtol=0)
                box = ts.dimensions.copy()
                assert np.isfinite(box).all() and np.linalg.det(triclinic_vectors(box)) > 0
                expected = saved.loc[row.saved_frame_index,residues.distance_column].to_numpy(dtype=float)
                original_pairs = distance_array(heavy.positions,ligand_heavy.positions,box=box)
                source_distances = np.minimum.reduceat(original_pairs.min(axis=1),starts)
                assert np.allclose(source_distances,expected,atol=FIGURE3_TOLERANCE_A,rtol=0)
                original_bond_vectors = selected.positions[bond_indices[:,1]]-selected.positions[bond_indices[:,0]]
                original_bond_lengths = np.linalg.norm(minimize_vectors(original_bond_vectors,box),axis=1)
                enzyme_xyz = make_whole(protein,inplace=False).astype(float)
                pha_xyz = make_whole(pha,inplace=False).astype(float)
                i,j = np.unravel_index(np.argmin(original_pairs),original_pairs.shape)
                delta = pha_xyz[pha_heavy_local[j]]-enzyme_xyz[protein_heavy_local[i]]
                shift = minimize_vectors(delta[None,:],box)[0]-delta
                lattice_coefficients = shift @ np.linalg.inv(triclinic_vectors(box).astype(float))
                assert np.allclose(lattice_coefficients,np.rint(lattice_coefficients),atol=2e-6,rtol=0)
                pha_xyz += shift  # Whole molecule; no adjustment of physical relative separation.
                snapshot.atoms.positions = np.vstack((enzyme_xyz,pha_xyz))
                snapshot.dimensions = box
                path = scratch / row.snapshot_filename
                with mda.Writer(str(path),n_atoms=len(selected),bonds="all",reindex=True) as writer:
                    writer.write(snapshot.atoms)
                # Record the actual saved mapping inside the standalone structure too.
                content = path.read_text()
                content = (f"REMARK 950 SYSTEM {system}\nREMARK 950 TIME_NS {row.time_ns:.8f} "
                           f"SAVED_XTC_FRAME {row.saved_frame_index}\n"
                           "REMARK 950 CHAIN A PROA=seg_0_PROA; CHAIN B LIG=seg_1_LIG\n"
                           "REMARK 950 WHOLE MOLECULES; PHA MOVED ONLY BY A PERIODIC LATTICE VECTOR\n" + content)
                path.write_text(content)
                exported = mda.Universe(str(path))
                try:
                    assert len(exported.atoms) == len(selected) and len(exported.trajectory) == 1
                    for attribute in ("names","resnames","resids"):
                        assert np.array_equal(getattr(exported.atoms,attribute),getattr(selected,attribute))
                    assert np.array_equal(exported.atoms.chainIDs,snapshot.atoms.chainIDs)
                    assert np.array_equal(exported.atoms.segids,snapshot.atoms.segids)
                    assert np.array_equal(exported.bonds.indices,snapshot.bonds.indices)
                    eh = exported.atoms[protein_heavy_local]
                    lh = exported.atoms[len(protein)+pha_heavy_local]
                    direct = np.minimum.reduceat(distance_array(eh.positions,lh.positions).min(axis=1),starts)
                    periodic = np.minimum.reduceat(distance_array(eh.positions,lh.positions,box=box).min(axis=1),starts)
                    # Verify the stored PDB box as well (CRYST1 rounds box lengths/angles).
                    pdb_periodic = np.minimum.reduceat(distance_array(eh.positions,lh.positions,
                                                        box=exported.dimensions).min(axis=1),starts)
                    expected_contacts = expected <= CONTACT_CUTOFF_A
                    for measured in (direct,periodic,pdb_periodic):
                        assert np.array_equal(measured <= CONTACT_CUTOFF_A,expected_contacts), (
                            f"{system} {row.time_ns:g} ns: exported contact classifications differ.")
                    assert np.allclose(periodic,expected,atol=0.002,rtol=0)
                    assert np.allclose(pdb_periodic,expected,atol=0.002,rtol=0)
                    assert np.isclose(direct.min(),expected.min(),atol=0.002,rtol=0)
                    bond_vectors = exported.atoms.positions[bond_indices[:,1]]-exported.atoms.positions[bond_indices[:,0]]
                    assert np.allclose(np.linalg.norm(bond_vectors,axis=1),original_bond_lengths,atol=0.002,rtol=0)
                    highlighted = exported.select_atoms(row.vmd_selection.replace("chain A","chainID A").replace("none","index -1"))
                    # Use MDAnalysis chainID syntax to verify the equivalent VMD chain selection.
                    listed = set(zip(highlighted.residues.resnames,highlighted.residues.resids))
                    expected_listed = set()
                    for residue in residues.loc[expected_contacts].itertuples():
                        token = residue_text(residue)
                        if token in row.contacting_leading_residues.split("; ") or (
                            row.contacting_leading_residues.startswith("Other residues") and
                            token in row.all_contacting_residues.split("; ")):
                            expected_listed.add((residue.resname,int(residue.resid)))
                    assert listed == expected_listed
                    validation.append({"source_min_distance_A":float(expected.min()),
                                       "snapshot_min_distance_A":float(direct.min()),
                                       "max_residue_distance_error_A":float(np.max(np.abs(periodic-expected))),
                                       "validated_residue_count":len(residues),
                                       "pha_image_shift_A":json.dumps(shift.tolist())})
                finally:
                    exported.trajectory.close()
                destination = paths["xtc"].parent / "representative_snapshots" / row.snapshot_filename
                payloads[destination] = content
            assert residue_input_stats(paths) == meta["input_size_mtime_ns"]
        finally:
            u.trajectory.close()
        result["snapshots"] = result["snapshots"].reset_index(drop=True)
        checks = pd.DataFrame(validation)
        for column in checks:
            result["snapshots"][column] = checks[column]
        result["snapshots"]["source_structure"] = str(structure)
        result["snapshots"]["source_topology"] = str(paths["tpr"])
        result["snapshots"]["source_trajectory"] = str(paths["xtc"])
        return payloads
else:
    print("EXPORT_REPRESENTATIVE_FRAMES=False; this section is disabled.")


In [ ]:
if RUN_PERIOD_TABLES or EXPORT_REPRESENTATIVE_FRAMES:
    from IPython.display import Markdown
    pdb_payloads = {}
    if EXPORT_REPRESENTATIVE_FRAMES:
        with TemporaryDirectory(prefix="ipha_representative_snapshots_") as scratch:
            for system, result in period_results.items():
                pdb_payloads.update(stage_system_snapshots(system, result, Path(scratch)))
                print(f"{system}: selected PDBs re-read; atom identities, contacts, separation and bonds validated.")

    csv_payloads = {}
    cutoff_tag = residue_cutoff_tag(CONTACT_CUTOFF_A)
    for system, result in period_results.items():
        prefix = residue_paths(system)["prefix"]
        tables = {"top10_overall": result["overall"], "top10_by_period": result["periods"],
                  "all_residues_overall": result["all_overall"], "all_residues_by_period": result["all_periods"],
                  ("snapshots_exported_by_period" if EXPORT_REPRESENTATIVE_FRAMES else "snapshots_by_period"): result["snapshots"]}
        for name, table in tables.items():
            table = table.copy()
            table.insert(0, "system", system.removesuffix("_gromacs"))
            csv_payloads[Path(f"{prefix}_residue_pha_{name}_{cutoff_tag}A.csv")] = table.to_csv(index=False)
        display(Markdown(f"**{SYSTEM_LABELS[system]} — leading residues and ANC45 active site**"))
        overall = result["all_overall"]
        shown = overall.loc[((overall['rank'] <= 10) & (overall.contact_time_ns > 0)) | overall.active_site].copy()
        shown["contact %"] = shown.occupancy_percent.map(contact_percentage_text)
        display(highlight_table(shown[["rank", "resname", "resid", "active_site", "contact %"]]))
        display(Markdown(r"\* Contact percentage greater than 10% (before rounding). Red rows = ANC45 active site."))
        for start, end in SUMMARY_WINDOWS_NS:
            table = result["all_periods"]
            shown = table.loc[(table.window_start_ns == start) &
                              (((table['rank'] <= 10) & (table.contact_time_ns > 0)) | table.active_site)].copy()
            shown["contact %"] = shown.occupancy_percent.map(contact_percentage_text)
            display(Markdown(f"**{start:g}–{end:g} ns**"))
            display(highlight_table(shown[["rank", "resname", "resid", "active_site", "contact %"]]))
        display(result["snapshots"][["time_window", "time_ns", "state", "contacting_leading_residues",
                                     "contacting_active_site_residues", "snapshot_filename", "vmd_selection",
                                     "active_site_vmd_selection"]])
        print("VMD reference:", system_inputs(system)["gro"])
        print("PDB status:", "validated for export" if EXPORT_REPRESENTATIVE_FRAMES else "export disabled; filenames are planned only")

    # Preflight every final destination before any write. Existing different outputs stop the whole save.
    payloads = {}
    if SAVE_PERIOD_RESULTS:
        payloads.update(csv_payloads)
    if EXPORT_REPRESENTATIVE_FRAMES:
        payloads.update(pdb_payloads)
    for path, content in payloads.items():
        if path.is_symlink() or path.parent.is_symlink():
            raise ValueError(f"Refuse a symlink output: {path}")
        if path.exists() and path.read_text() != content:
            raise FileExistsError(f"Existing different result protected: {path}; choose a new OUTPUT_TAG.")
    for path, content in payloads.items():
        path.parent.mkdir(parents=True, exist_ok=True)
        if not path.exists():
            with path.open("x") as handle:
                handle.write(content)
        assert path.read_text() == content
    print(f"Saved/reused {len(payloads)} protected outputs. Ranking saves={SAVE_PERIOD_RESULTS}; PDB export={EXPORT_REPRESENTATIVE_FRAMES}.")
else:
    print("Period tables and representative-frame exports disabled.")


## Running this notebook

**First time:** select **ipha_clean**, set **`RUN_ANALYSIS=True`**, then **Run All**.

**Later:** set **`RUN_ANALYSIS=False`**, then **Run All** to redraw from saved CSVs.

**Optional PDBs:** also set **`EXPORT_REPRESENTATIVE_FRAMES=True`**.

No analysis or notebook cells were executed during preparation. The original notebook and existing results remain unchanged.
